In [1]:
import os

# Create the package directories
os.makedirs('FASEROH/data', exist_ok=True)
os.makedirs('FASEROH/models', exist_ok=True)
os.makedirs('FASEROH/training', exist_ok=True)
os.makedirs('FASEROH/evaluation', exist_ok=True)
os.makedirs('FASEROH/working', exist_ok=True)

# Create __init__.py files to make them valid Python packages
paths = ['FASEROH', 'FASEROH/data', 'FASEROH/models', 'FASEROH/training', 'FASEROH/evaluation']
for path in paths:
    with open(os.path.join(path, '__init__.py'), 'w') as f:
        pass

!touch /kaggle/working/FASEROH/working/sample_predictions.json

In [ ]:
%%writefile FASEROH/data/tokenizer.py

import re
import pickle
from typing import List, Dict, Tuple, Optional, Union, Set, Iterable
from dataclasses import dataclass, field, asdict
import sympy as sp
from sympy.printing import StrPrinter
from sympy.parsing.sympy_parser import parse_expr
import numpy as np
from collections import Counter
from pathlib import Path
import logging
from tqdm import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class TokenizerConfig:
    """Configuration for mathematical expression tokenizer"""
    # Special tokens
    pad_token: str = "<PAD>"
    unk_token: str = "<UNK>"
    start_token: str = "<START>"
    end_token: str = "<END>"
    
    # Multi-character mathematical tokens (domain-specific)
    # This reduces sequence length and captures mathematical semantics better than pure char-level
    math_tokens: List[str] = field(default_factory=lambda: [
        "sin", "cos", "tan", "exp", "log", "sqrt",  # Functions
        "**",  # Power operator (Python syntax)
        "pi", "E",  # Constants
    ])
    
    # Character set for basic tokenization
    allowed_chars: Set[str] = field(default_factory=lambda: set(
        "x0123456789+-*/()^.,= "
    ))
    
    max_input_length: int = 50
    max_output_length: int = 100
    
    # SymPy canonicalization settings
    expand_expression: bool = True
    simplify_rational: bool = True
    sort_terms: bool = True  # Ascending powers

class MathematicalTokenizer:
    """
    Intelligent tokenizer for mathematical expressions.
    Uses hybrid approach: multi-char tokens for functions, char-level for rest.
    """
    
    def __init__(self, config: Optional[TokenizerConfig] = None):
        self.config = config or TokenizerConfig()
        
        # Vocabulary mappings
        self.token_to_idx: Dict[str, int] = {}
        self.idx_to_token: Dict[int, str] = {}
        self.vocab_size: int = 0
        
        # Special token IDs
        self.pad_id: int = 0
        self.unk_id: int = 1
        self.start_id: int = 2
        self.end_id: int = 3
        
        # Compiled regex patterns for tokenization
        self._compile_patterns()
        
        self.is_fitted = False
    
    def _compile_patterns(self):
        """Compile regex patterns for efficient tokenization"""
        # Sort math tokens by length (descending) to match longest first
        sorted_math_tokens = sorted(self.config.math_tokens, key=len, reverse=True)
        
        # Build pattern: match multi-char tokens OR single allowed chars
        math_pattern = '|'.join(re.escape(tok) for tok in sorted_math_tokens)
        char_pattern = '[' + re.escape(''.join(self.config.allowed_chars)) + ']'
        
        # Combined pattern: either math token or single character
        self.token_pattern = re.compile(f'({math_pattern}|{char_pattern})')
        
        # Pattern to identify numbers (including decimals and negatives)
        self.number_pattern = re.compile(r'-?\d+\.?\d*')
    
    def _canonicalize_expression(self, expr_str: str) -> str:
        """
        Canonicalize mathematical expression using SymPy.
        Ensures consistent formatting: ascending powers, simplified fractions.
        """
        try:
            # Parse string to SymPy expression
            x = sp.Symbol('x')
            # Replace ^ with ** for Python compatibility if needed
            expr_str = expr_str.replace('^', '**')
            
            expr = parse_expr(expr_str, local_dict={'x': x, 'sin': sp.sin, 'cos': sp.cos, 
                                                     'exp': sp.exp, 'log': sp.log, 
                                                     'sqrt': sp.sqrt, 'tan': sp.tan})
            
            # Expand expression (distribute products over sums)
            if self.config.expand_expression:
                expr = sp.expand(expr)
            
            # Simplify rational numbers
            if self.config.simplify_rational:
                expr = sp.nsimplify(expr, [sp.pi, sp.E])
                expr = sp.simplify(expr)
            
            # Convert back to string with consistent formatting
            # result = str(expr)

            printer = StrPrinter({'order': 'rev-lex'})
            result = printer.doprint(expr)
            
            # Normalize whitespace
            result = ' '.join(result.split())
            
            return result
            
        except Exception as e:
            logger.warning(f"Canonicalization failed for '{expr_str}': {e}")
            # Return cleaned original if parsing fails
            return expr_str.replace('^', '**').strip()
    
    def tokenize(self, expression: str, canonicalize: bool = True) -> List[str]:
        """
        Tokenize mathematical expression into list of tokens.
        
        Args:
            expression: Mathematical expression string
            canonicalize: Whether to canonicalize using SymPy first
            
        Returns:
            List of tokens
        """
        if canonicalize:
            expression = self._canonicalize_expression(expression)
        
        # Insert spaces around operators to ensure proper splitting
        # But preserve multi-char operators like **
        expr_processed = expression
        
        # Find all tokens using regex
        tokens = self.token_pattern.findall(expr_processed)
        
        # Filter out empty strings and whitespace-only tokens
        tokens = [t.strip() for t in tokens if t.strip()]
        
        return tokens

    def fit(self, expressions: Iterable[str], min_freq: int = 1):
        """
        Build vocabulary from an iterable of expressions (supports streaming).
        
        Args:
            expressions: Iterable/Generator of mathematical expression strings
            min_freq: Minimum frequency for token to be included
        """
        logger.info("Building vocabulary (streaming mode)...")
        
        # Count token frequencies
        token_counter = Counter()
        
        # Stream processing: Only holds one equation in RAM at a time
        for i, expr in enumerate(expressions):
            # --- THE FIX: canonicalize=False ---
            # SymPy caches parsed trees. Re-parsing 65k strings causes an OOM crash.
            # The Phase 1 data generator already formatted these perfectly!
            tokens = self.tokenize(expr, canonicalize=False)
            token_counter.update(tokens)
            
            # Print a heartbeat so you know it hasn't frozen!
            if (i + 1) % 10000 == 0:
                logger.info(f"  Scanned {i + 1} expressions...")
        
        # Initialize vocabulary with special tokens
        self.token_to_idx = {
            self.config.pad_token: self.pad_id,
            self.config.unk_token: self.unk_id,
            self.config.start_token: self.start_id,
            self.config.end_token: self.end_id,
        }
        
        # Add frequent tokens to vocabulary
        idx = len(self.token_to_idx)
        for token, freq in token_counter.most_common():
            if freq >= min_freq and token not in self.token_to_idx:
                self.token_to_idx[token] = idx
                idx += 1
        
        # Create reverse mapping
        self.idx_to_token = {v: k for k, v in self.token_to_idx.items()}
        self.vocab_size = len(self.token_to_idx)
        self.is_fitted = True
        
        logger.info(f"Vocabulary built: {self.vocab_size} tokens")
        logger.info(f"Top 10 tokens: {token_counter.most_common(10)}")
        
        return self
    
    def encode(self, expression: str, add_special_tokens: bool = True, 
               max_length: Optional[int] = None) -> List[int]:
        """
        Encode expression to list of integer indices.
        
        Args:
            expression: Mathematical expression string
            add_special_tokens: Whether to add <START> and <END>
            max_length: Maximum sequence length (truncate if longer)
            
        Returns:
            List of integer token IDs
        """
        if not self.is_fitted:
            raise RuntimeError("Tokenizer must be fitted before encoding")
        
        tokens = self.tokenize(expression, canonicalize = False)
        
        # Convert to IDs
        token_ids = [self.token_to_idx.get(tok, self.unk_id) for tok in tokens]
        
        # Add special tokens
        if add_special_tokens:
            token_ids = [self.start_id] + token_ids + [self.end_id]
        
        # Truncate if necessary
        if max_length and len(token_ids) > max_length:
            token_ids = token_ids[:max_length-1] + [self.end_id]
        
        return token_ids
    
    def decode(self, token_ids: List[int], skip_special_tokens: bool = True) -> str:
        """
        Decode list of integer indices back to string.
        
        Args:
            token_ids: List of token IDs
            skip_special_tokens: Whether to remove <PAD>, <START>, <END>, <UNK>
            
        Returns:
            Reconstructed expression string
        """
        tokens = []
        for idx in token_ids:
            token = self.idx_to_token.get(idx, self.config.unk_token)
            
            if skip_special_tokens and token in [
                self.config.pad_token, self.config.start_token, 
                self.config.end_token, self.config.unk_token
            ]:
                continue
            
            tokens.append(token)
        
        # Join with spaces and cleanup
        result = ''.join(tokens)
        # Fix spacing around operators for readability
        result = result.replace('**', '^')  # Convert back to caret for readability if desired
        
        return result
    
    def encode_batch(self, expressions: List[str], max_length: int,
                     pad_to_max: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        """
        Encode batch of expressions with padding and masking.
        
        Returns:
            Tuple of (padded_sequences, attention_masks)
            padded_sequences: (batch_size, max_length) array of token IDs
            attention_masks: (batch_size, max_length) binary array (1=real token, 0=pad)
        """
        sequences = []
        lengths = []
        
        # for expr in expressions:
        #     seq = self.encode(expr, add_special_tokens=True, max_length=max_length)
        #     sequences.append(seq)
        #     lengths.append(len(seq))

        # --- ADDED TQDM TO THE SILENT ENCODING LOOP ---
        for expr in tqdm(expressions, desc="Encoding Batch to Tensors"):
            seq = self.encode(expr, add_special_tokens=True, max_length=max_length)
            sequences.append(seq)
            lengths.append(len(seq))
        # ----------------------------------------------
        
        # Padding
        if pad_to_max:
            padded = np.full((len(sequences), max_length), self.pad_id, dtype=np.int32)
            masks = np.zeros((len(sequences), max_length), dtype=np.int32)
            
            for i, seq in enumerate(sequences):
                end_idx = min(len(seq), max_length)
                padded[i, :end_idx] = seq[:end_idx]
                masks[i, :end_idx] = 1  # 1 for real tokens, 0 for pad
        
        return padded, masks
    
    def save(self, path: Union[str, Path]):
        """Save tokenizer vocabulary and config"""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        
        state = {
            'token_to_idx': self.token_to_idx,
            'idx_to_token': self.idx_to_token,
            'config': {
                'pad_token': self.config.pad_token,
                'unk_token': self.config.unk_token,
                'start_token': self.config.start_token,
                'end_token': self.config.end_token,
                'math_tokens': self.config.math_tokens,
                'allowed_chars': list(self.config.allowed_chars),
                'max_input_length': self.config.max_input_length,
                'max_output_length': self.config.max_output_length,
            },
            'vocab_size': self.vocab_size,
            'is_fitted': self.is_fitted
        }
        
        with open(path, 'wb') as f:
            pickle.dump(state, f)
        
        logger.info(f"Tokenizer saved to {path}")
    
    @classmethod
    def load(cls, path: Union[str, Path]) -> 'MathematicalTokenizer':
        """Load tokenizer from disk"""
        with open(path, 'rb') as f:
            state = pickle.load(f)
        
        # Reconstruct config
        config = TokenizerConfig(
            pad_token=state['config']['pad_token'],
            unk_token=state['config']['unk_token'],
            start_token=state['config']['start_token'],
            end_token=state['config']['end_token'],
            math_tokens=state['config']['math_tokens'],
            allowed_chars=set(state['config']['allowed_chars']),
            max_input_length=state['config']['max_input_length'],
            max_output_length=state['config']['max_output_length'],
        )
        
        tokenizer = cls(config)
        tokenizer.token_to_idx = state['token_to_idx']
        tokenizer.idx_to_token = {int(k): v for k, v in state['idx_to_token'].items()}
        tokenizer.vocab_size = state['vocab_size']
        tokenizer.is_fitted = state['is_fitted']
        tokenizer._compile_patterns()
        
        return tokenizer

Writing FASEROH/data/tokenizer.py


In [ ]:
%%writefile FASEROH/data/dataset.py

import json
import re
import pickle
from typing import List, Dict, Tuple, Optional, Union, Set
from dataclasses import dataclass, field
from pathlib import Path
import logging
from collections import Counter, defaultdict
import numpy as np
from tqdm import tqdm

import sympy as sp
from sympy.parsing.sympy_parser import parse_expr
from sympy.printing import StrPrinter
import sys

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

import os
sys.path.append(os.getcwd())

from FASEROH.data.tokenizer import MathematicalTokenizer, TokenizerConfig

import numpy as np

class NumpyEncoder(json.JSONEncoder):
    """Custom encoder for numpy data types"""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

class TaylorDatasetPipeline:
    """
    Complete data pipeline for Taylor expansion dataset.
    Handles loading, tokenization, batching, and masking.
    """
    
    def __init__(self, 
                 input_tokenizer: MathematicalTokenizer,
                 output_tokenizer: Optional[MathematicalTokenizer] = None,
                 max_input_len: int = 50,
                 max_output_len: int = 100):
        """
        Initialize pipeline.
        
        Args:
            input_tokenizer: Tokenizer for input functions
            output_tokenizer: Tokenizer for Taylor expansions (can be same as input)
            max_input_len: Maximum input sequence length
            max_output_len: Maximum output sequence length
        """
        self.input_tokenizer = input_tokenizer
        self.output_tokenizer = output_tokenizer or input_tokenizer
        self.max_input_len = max_input_len
        self.max_output_len = max_output_len
        
        self.train_data: List[Dict] = []
        self.test_data: List[Dict] = []
        self.metadata: Dict = {}
    
    def load_from_jsonl(self, train_path: Path, test_path: Path):
        """Load dataset from JSONL files generated in Phase 1"""
        logger.info(f"Loading data from {train_path} and {test_path}")
        
        def load_jsonl(path):
            data = []
            with open(path, 'r') as f:
                for line in f:
                    data.append(json.loads(line.strip()))
            return data
        
        self.train_data = load_jsonl(train_path)
        self.test_data = load_jsonl(test_path)
        
        logger.info(f"Loaded {len(self.train_data)} training samples, {len(self.test_data)} test samples")
        
        return self
    
    def prepare_sequence_pairs(self, split: str = 'train') -> Tuple[List[str], List[str]]:
        """
        Extract input-output string pairs from loaded data.
        
        Returns:
            Tuple of (input_expressions, target_expressions)
        """
        data = self.train_data if split == 'train' else self.test_data
        
        inputs = []
        targets = []
        
        for item in data:
            # Input: mathematical function
            inputs.append(item['input_func'])
            # Target: Taylor expansion
            targets.append(item['taylor_expansion'])
        
        return inputs, targets
    
    def create_tf_dataset(self, split: str = 'train', batch_size: int = 512):
        """
        Create TensorFlow dataset with proper masking.
        For use with Keras/TF models.
        """
        try:
            import tensorflow as tf
        except ImportError:
            raise ImportError("TensorFlow is required. Install with: pip install tensorflow")
        
        inputs, targets = self.prepare_sequence_pairs(split)
        
        # Encode sequences
        encoder_input, encoder_mask = self.input_tokenizer.encode_batch(
            inputs, self.max_input_len
        )
        decoder_target, _ = self.output_tokenizer.encode_batch(
            targets, self.max_output_len
        )
        
        # Create decoder input (shifted right by 1, start with <START>)
        decoder_input = np.zeros_like(decoder_target)
        decoder_input[:, 0] = self.output_tokenizer.start_id
        decoder_input[:, 1:] = decoder_target[:, :-1]
        
        # Create dataset
        dataset = tf.data.Dataset.from_tensor_slices({
            'encoder_input': encoder_input,
            'encoder_mask': encoder_mask,
            'decoder_input': decoder_input,
            'decoder_target': decoder_target
        })
        
        dataset = dataset.shuffle(buffer_size=1000).batch(batch_size)
        
        return dataset
    
    def create_pytorch_dataset(self, split: str = 'train'):
        """
        Create PyTorch dataset with padding masks.
        For use with PyTorch/Transformer models.
        """
        try:
            import torch
            from torch.utils.data import Dataset, DataLoader
        except ImportError:
            raise ImportError("PyTorch is required. Install with: pip install torch")
        
        inputs, targets = self.prepare_sequence_pairs(split)
        
        # Encode
        encoder_input, encoder_mask = self.input_tokenizer.encode_batch(
            inputs, self.max_input_len
        )
        decoder_target, decoder_mask = self.output_tokenizer.encode_batch(
            targets, self.max_output_len
        )
        
        # Create decoder input (teacher forcing)
        decoder_input = np.zeros_like(decoder_target)
        decoder_input[:, 0] = self.output_tokenizer.start_id
        decoder_input[:, 1:] = decoder_target[:, :-1]
        
        class TaylorDataset(Dataset):
            def __init__(self, enc_inp, enc_mask, dec_inp, dec_tgt):
                self.enc_inp = torch.LongTensor(enc_inp)
                self.enc_mask = torch.BoolTensor(enc_mask)
                self.dec_inp = torch.LongTensor(dec_inp)
                self.dec_tgt = torch.LongTensor(dec_tgt)
            
            def __len__(self):
                return len(self.enc_inp)
            
            def __getitem__(self, idx):
                return {
                    'encoder_input': self.enc_inp[idx],
                    'encoder_mask': self.enc_mask[idx],
                    'decoder_input': self.dec_inp[idx],
                    'decoder_target': self.dec_tgt[idx]
                }
        
        dataset = TaylorDataset(encoder_input, encoder_mask, decoder_input, decoder_target)
        
        return dataset
    
    def create_lytorch_dataloader(self, split: str = 'train', batch_size: int = 512, 
                                   shuffle: bool = True):
        """Convenience method to get PyTorch DataLoader"""
        try:
            from torch.utils.data import DataLoader
        except ImportError:
            raise ImportError("PyTorch is required")
        
        dataset = self.create_pytorch_dataset(split)
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    
    def analyze_sequence_lengths(self) -> Dict:
        """Analyze length distribution to validate max_length choices"""
        inputs, targets = self.prepare_sequence_pairs('train')
        
        input_lengths = []
        target_lengths = []

        logger.info("Analyzing sequence lengths...")
        for inp, tgt in tqdm(zip(inputs, targets), total=len(inputs), desc="Calculating Max Lengths"):
            # canonicalize=False stops the SymPy RAM freeze!
            inp_len = len(self.input_tokenizer.tokenize(inp, canonicalize=False))
            tgt_len = len(self.output_tokenizer.tokenize(tgt, canonicalize=False))
            input_lengths.append(inp_len)
            target_lengths.append(tgt_len)
        
        stats = {
            'input': {
                'mean': np.mean(input_lengths),
                'std': np.std(input_lengths),
                'max': np.max(input_lengths),
                'percentile_95': np.percentile(input_lengths, 95),
                'percentile_99': np.percentile(input_lengths, 99)
            },
            'target': {
                'mean': np.mean(target_lengths),
                'std': np.std(target_lengths),
                'max': np.max(target_lengths),
                'percentile_95': np.percentile(target_lengths, 95),
                'percentile_99': np.percentile(target_lengths, 99)
            }
        }
        
        logger.info("Sequence Length Analysis:")
        logger.info(f"Input - Mean: {stats['input']['mean']:.1f}, "
                   f"95th %ile: {stats['input']['percentile_95']:.1f}, "
                   f"Max: {stats['input']['max']}")
        logger.info(f"Target - Mean: {stats['target']['mean']:.1f}, "
                   f"95th %ile: {stats['target']['percentile_95']:.1f}, "
                   f"Max: {stats['target']['max']}")
        
        return stats

def build_complete_pipeline(train_jsonl: str, test_jsonl: str, 
                           output_dir: str = "/kaggle/working/tokenized_data"):
    """
    Complete pipeline: Load Phase 1 data, fit tokenizers, save processed datasets.
    
    Args:
        train_jsonl: Path to training JSONL file
        test_jsonl: Path to test JSONL file
        output_dir: Directory to save tokenized outputs
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load raw data to analyze
    train_data = []
    with open(train_jsonl, 'r') as f:
        for line in f:
            train_data.append(json.loads(line.strip()))
    
    # Extract text
    input_exprs = [d['input_func'] for d in train_data]
    target_exprs = [d['taylor_expansion'] for d in train_data]
    
    # Create and fit input tokenizer
    logger.info("Fitting input tokenizer...")
    input_tokenizer = MathematicalTokenizer()
    input_tokenizer.fit(input_exprs, min_freq=1)
    input_tokenizer.save(output_dir / "input_tokenizer.pkl")
    
    # Create and fit output tokenizer (can share vocabulary with input)
    logger.info("Fitting output tokenizer...")
    output_tokenizer = MathematicalTokenizer()
    # Fit on both inputs and targets to ensure shared vocabulary
    all_exprs = input_exprs + target_exprs
    output_tokenizer.fit(all_exprs, min_freq=1)
    output_tokenizer.save(output_dir / "output_tokenizer.pkl")
    
    logger.info(f"Input vocab size: {input_tokenizer.vocab_size}")
    logger.info(f"Output vocab size: {output_tokenizer.vocab_size}")
    
    # Initialize pipeline
    pipeline = TaylorDatasetPipeline(
        input_tokenizer=input_tokenizer,
        output_tokenizer=output_tokenizer,
        max_input_len=50,
        max_output_len=100
    )
    pipeline.load_from_jsonl(Path(train_jsonl), Path(test_jsonl))
    
    # Analyze lengths to validate max_length parameters
    stats = pipeline.analyze_sequence_lengths()
    
    # Adjust max lengths based on analysis if needed
    recommended_input_len = int(stats['input']['percentile_95']) + 2  # +2 for START/END
    recommended_output_len = int(stats['target']['percentile_95']) + 2
    
    logger.info(f"Recommended max_input_len: {recommended_input_len}")
    logger.info(f"Recommended max_output_len: {recommended_output_len}")
    
    # Save processed numpy arrays for fast loading
    logger.info("Processing and saving datasets...")
    
    for split in ['train', 'test']:
        inputs, targets = pipeline.prepare_sequence_pairs(split)
        
        # Encode
        enc_inp, enc_mask = input_tokenizer.encode_batch(inputs, max_length=50)
        dec_tgt, dec_mask = output_tokenizer.encode_batch(targets, max_length=100)
        
        # Create decoder input (shifted)
        dec_inp = np.zeros_like(dec_tgt)
        dec_inp[:, 0] = output_tokenizer.start_id
        dec_inp[:, 1:] = dec_tgt[:, :-1]
        
        # Save
        np.savez(
            output_dir / f"{split}_data.npz",
            encoder_input=enc_inp,
            encoder_mask=enc_mask,
            decoder_input=dec_inp,
            decoder_target=dec_tgt,
            decoder_mask=dec_mask
        )
        
        logger.info(f"Saved {split} set: {len(inputs)} samples")
    
    # Save metadata
    metadata = {
        'vocab_size_input': input_tokenizer.vocab_size,
        'vocab_size_output': output_tokenizer.vocab_size,
        'max_input_length': 50,
        'max_output_length': 100,
        'train_samples': len(pipeline.train_data),
        'test_samples': len(pipeline.test_data),
        'sequence_stats': stats,
        'special_tokens': {
            'pad': input_tokenizer.pad_id,
            'unk': input_tokenizer.unk_id,
            'start': input_tokenizer.start_id,
            'end': input_tokenizer.end_id
        }
    }
    
    with open(output_dir / "metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2, cls=NumpyEncoder)
    
    logger.info(f"Pipeline complete. Data saved to {output_dir}")
    
    return pipeline, metadata

# Example usage and testing
if __name__ == "__main__":
    # Example: Test tokenization
    tokenizer = MathematicalTokenizer()
    
    test_exprs = [
        "sin(x) + x**2",
        "exp(-x)*cos(2*x)",
        "x*log(1+x**2)",
        "1 + x + x**2/2 - x**3/6"
    ]
    
    print("Testing Tokenization:")
    print("=" * 60)
    
    for expr in test_exprs:
        tokens = tokenizer.tokenize(expr, canonicalize=True)
        print(f"\nInput:  {expr}")
        print(f"Tokens: {tokens}")
        
        # Simulate encoding (need to fit first for real IDs)
        if tokenizer.is_fitted:
            ids = tokenizer.encode(expr)
            decoded = tokenizer.decode(ids)
            print(f"IDs:    {ids}")
            print(f"Back:   {decoded}")
    
    # Fit on test data
    tokenizer.fit(test_exprs)
    
    print("\n" + "=" * 60)
    print("After Fitting:")
    print("=" * 60)
    
    for expr in test_exprs:
        tokens = tokenizer.tokenize(expr)
        ids = tokenizer.encode(expr)
        decoded = tokenizer.decode(ids)
        print(f"\nInput:  {expr}")
        print(f"Tokens: {tokens}")
        print(f"IDs:    {ids}")
        print(f"Back:   {decoded}")
    
    # Test batch encoding
    padded, masks = tokenizer.encode_batch(test_exprs, max_length=20)
    print(f"\nBatch shape: {padded.shape}")
    print(f"Mask shape:  {masks.shape}")
    print(f"First sample IDs: {padded[0]}")
    print(f"First sample mask: {masks[0]}")

Writing FASEROH/data/dataset.py


In [ ]:
%%writefile FASEROH/evaluation/evaluate.py

import torch
import numpy as np
import json
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr
from typing import Dict, List, Tuple, Optional, Union, Any
from dataclasses import dataclass, field
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import pandas as pd
import logging
from tqdm import tqdm
import warnings

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class EvaluationResult:
    """Container for comprehensive evaluation results"""
    # Basic metrics
    token_accuracy: float = 0.0
    sequence_accuracy: float = 0.0
    bleu_score: float = 0.0
    perplexity: float = 0.0
    
    # Mathematical correctness (symbolic)
    symbolic_accuracy: float = 0.0
    symbolic_equivalence_rate: float = 0.0  # Syntactically different but mathematically equal
    
    # Categorized metrics
    by_complexity: Dict[str, Dict[str, float]] = field(default_factory=dict)
    by_base_function: Dict[str, Dict[str, float]] = field(default_factory=dict)
    
    # Error analysis
    total_samples: int = 0
    correct_predictions: int = 0
    symbolic_matches: int = 0
    equivalence_matches: int = 0
    
    # Detailed results for visualization
    predictions: List[Dict] = field(default_factory=list)
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            'token_accuracy': self.token_accuracy,
            'sequence_accuracy': self.sequence_accuracy,
            'bleu_score': self.bleu_score,
            'perplexity': self.perplexity,
            'symbolic_accuracy': self.symbolic_accuracy,
            'symbolic_equivalence_rate': self.symbolic_equivalence_rate,
            'by_complexity': self.by_complexity,
            'by_base_function': self.by_base_function,
            'total_samples': self.total_samples,
            'correct_predictions': self.correct_predictions,
            'symbolic_matches': self.symbolic_matches,
        }


class SymbolicVerifier:
    """
    Verifies mathematical equivalence using SymPy.
    Handles cases like 1/2*x vs x/2 that string comparison misses.
    """
    
    def __init__(self, tolerance: float = 1e-10):
        self.x = sp.Symbol('x')
        self.tolerance = tolerance
        
    def parse_expression(self, expr_str: str) -> Optional[sp.Expr]:
        """Safely parse string to SymPy expression"""
        try:
            # Clean up common formatting issues
            expr_str = expr_str.strip()
            expr_str = expr_str.replace('^', '**')
            expr_str = expr_str.replace('<START>', '').replace('<END>', '').replace('<PAD>', '')
            expr_str = expr_str.replace('<UNK>', '')
            
            # Handle implicit multiplication (e.g., "2x" -> "2*x")
            # This is a simplified version; real implementation would need more robust parsing
            expr_str = expr_str.replace(')(', ')*(')
            
            return parse_expr(expr_str, local_dict={'x': self.x, 'X': self.x})
        except Exception as e:
            logger.debug(f"Failed to parse '{expr_str}': {e}")
            return None
    
    def check_equivalence(self, pred_str: str, truth_str: str) -> Tuple[bool, bool]:
        """
        Check mathematical equivalence between two expressions.
        
        Returns:
            (exact_match, symbolic_equivalence)
            exact_match: Syntactically identical (after normalization)
            symbolic_equivalence: Mathematically equal (simplify(pred - truth) == 0)
        """
        # String-level exact match (after basic normalization)
        pred_normalized = ' '.join(pred_str.lower().split())
        truth_normalized = ' '.join(truth_str.lower().split())
        exact_match = pred_normalized == truth_normalized
        
        # Symbolic verification
        pred_expr = self.parse_expression(pred_str)
        truth_expr = self.parse_expression(truth_str)
        
        if pred_expr is None or truth_expr is None:
            return exact_match, False
        
        try:
            # Check difference is zero
            diff = sp.simplify(pred_expr - truth_expr)
            symbolic_match = diff == 0
            
            # Additional numerical verification for safety
            if not symbolic_match:
                # Test at several points
                test_points = [0.1, 0.5, 1.0, -0.1, -0.5]
                numerical_match = True
                for pt in test_points:
                    try:
                        pred_val = float(pred_expr.subs(self.x, pt).evalf())
                        truth_val = float(truth_expr.subs(self.x, pt).evalf())
                        if abs(pred_val - truth_val) > self.tolerance:
                            numerical_match = False
                            break
                    except:
                        numerical_match = False
                        break
                
                # If numerically matches but not symbolically, it's an equivalent form
                equivalent_form = numerical_match and not symbolic_match
            else:
                equivalent_form = False
            
            return exact_match, (symbolic_match or equivalent_form)
            
        except Exception as e:
            logger.debug(f"Symbolic comparison failed: {e}")
            return exact_match, False


class AttentionVisualizer:
    """
    Generates attention heatmaps for LSTM and Transformer models.
    Shows alignment between input and output tokens.
    """
    
    def __init__(self, tokenizer, save_dir: str = "/kaggle/working/plots/"):
        self.tokenizer = tokenizer
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)

    def extract_attention_weights(self, model, batch: Dict[str, torch.Tensor]) -> Optional[np.ndarray]:
        """Extract attention weights from model during forward pass"""
        model.eval()
        
        with torch.no_grad():
            # 1. Dynamically find the model's device
            device = next(model.parameters()).device
            
            # 2. Extract and immediately push to GPU
            src = batch['encoder_input'].to(device)
            
            # Use get() safely, then push to device if it exists
            tgt_raw = batch.get('decoder_input', batch.get('decoder_target'))
            tgt = tgt_raw.to(device) if tgt_raw is not None else None
            
            if hasattr(model, 'decoder') and hasattr(model.decoder, 'enable_attention_storage'):
                model.decoder.enable_attention_storage(True)
                
                # src and tgt are now safely on cuda:0
                _ = model(src, tgt) 
                
                attn_matrix = model.decoder.get_attention_matrix()
                model.decoder.enable_attention_storage(False)
                
                # Ensure we return it as a numpy array for visualization
                if isinstance(attn_matrix, torch.Tensor):
                    return attn_matrix.cpu().numpy()
                return attn_matrix
            
            elif hasattr(model, 'decoder_layers'):
                # src and tgt are already on cuda:0 from step 2
                
                # Ensure your model's forward returns (logits, attn_weights)
                _, attn_weights = model(src, tgt, return_attention=True) 
                return attn_weights.cpu().numpy()
            
            else:
                return None
    
    def create_heatmap(self, attention: np.ndarray, 
                       src_tokens: List[str], 
                       tgt_tokens: List[str],
                       title: str = "Attention Alignment",
                       save_path: Optional[str] = None) -> plt.Figure:
        """
        Create attention heatmap visualization.
        
        Args:
            attention: (tgt_len, src_len) attention weights
            src_tokens: Source token strings
            tgt_tokens: Target token strings
            title: Plot title
            save_path: Where to save figure
        """
        fig, ax = plt.subplots(figsize=(max(8, len(src_tokens) * 0.5), 
                                      max(6, len(tgt_tokens) * 0.4)))
        
        # Create heatmap
        sns.heatmap(
            attention,
            xticklabels=src_tokens,
            yticklabels=tgt_tokens,
            cmap='viridis',
            annot=False,
            fmt='.2f',
            cbar_kws={'label': 'Attention Weight'},
            ax=ax,
            square=True
        )
        
        ax.set_xlabel('Input Function Tokens', fontsize=12)
        ax.set_ylabel('Taylor Expansion Tokens', fontsize=12)
        ax.set_title(title, fontsize=14, pad=20)
        
        # Rotate labels for readability
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            logger.info(f"Saved attention visualization to {save_path}")
        
        return fig
    
    def visualize_sample(self, model, batch: Dict, idx: int = 0,
                        save_name: Optional[str] = None) -> Optional[plt.Figure]:
        """
        Visualize attention for a single sample in batch.
        
        Args:
            model: The seq2seq model
            batch: Data batch
            idx: Index in batch to visualize
            save_name: Filename for saving
        """
        attention = self.extract_attention_weights(model, batch)
        
        if attention is None:
            logger.warning("Could not extract attention weights from model")
            return None
        
        # Get single sample attention
        if len(attention.shape) == 3:  # (batch, tgt_len, src_len)
            sample_attn = attention[idx]
        else:
            sample_attn = attention
        
        # Decode tokens
        src_ids = batch['encoder_input'][idx].cpu().numpy()
        tgt_ids = batch.get('decoder_input', batch.get('decoder_target'))[idx].cpu().numpy()
        
        src_tokens = [self.tokenizer.decode([t], skip_special_tokens=False) for t in src_ids]
        tgt_tokens = [self.tokenizer.decode([t], skip_special_tokens=False) for t in tgt_ids]
        
        # Clean up tokens for display
        src_tokens = [t for t in src_tokens if t not in ['<PAD>', '']]
        tgt_tokens = [t for t in tgt_tokens if t not in ['<PAD>', '']]
        
        # Truncate attention matrix to match cleaned tokens
        sample_attn = sample_attn[:len(tgt_tokens), :len(src_tokens)]
        
        title = f"Attention: {' '.join(src_tokens[:10])}..."
        if save_name is None:
            save_name = f"attention_sample_{idx}.png"
        
        save_path = self.save_dir / save_name
        
        return self.create_heatmap(sample_attn, src_tokens, tgt_tokens, title, str(save_path))


class CategorizedEvaluator:
    """
    Evaluates model performance broken down by categories.
    Identifies strengths and weaknesses across function types.
    """
    
    def __init__(self, symbolic_verifier: SymbolicVerifier):
        self.verifier = symbolic_verifier
        self.results_by_category = defaultdict(lambda: {
            'total': 0,
            'correct': 0,
            'symbolic_correct': 0,
            'token_correct': 0,
            'token_total': 0
        })
    
    def categorize_sample(self, sample: Dict) -> Tuple[str, List[str]]:
        """
        Determine categories for a sample.
        Returns (complexity_class, base_functions).
        """
        complexity = sample.get('complexity_class', 'unknown')
        base_funcs = sample.get('base_functions', [])
        
        # Normalize base functions
        if isinstance(base_funcs, str):
            base_funcs = [base_funcs]
        
        return complexity, base_funcs
    
    def update(self, sample: Dict, pred_str: str, truth_str: str, 
               token_acc: float, exact_match: bool, symbolic_match: bool):
        """Update category statistics"""
        complexity, base_funcs = self.categorize_sample(sample)
        
        # Update by complexity
        cat = self.results_by_category[complexity]
        cat['total'] += 1
        cat['correct'] += int(exact_match)
        cat['symbolic_correct'] += int(symbolic_match)
        cat['token_correct'] += int(token_acc * 100)  # Store as percentage sum
        cat['token_total'] += 1
        
        # Update by base function
        for func in base_funcs:
            cat = self.results_by_category[f"func_{func}"]
            cat['total'] += 1
            cat['correct'] += int(exact_match)
            cat['symbolic_correct'] += int(symbolic_match)
            cat['token_correct'] += int(token_acc * 100)
            cat['token_total'] += 1
    
    def get_results(self) -> Dict[str, Dict[str, float]]:
        """Calculate final metrics by category"""
        results = {}
        
        for category, stats in self.results_by_category.items():
            if stats['total'] > 0:
                results[category] = {
                    'sequence_accuracy': stats['correct'] / stats['total'],
                    'symbolic_accuracy': stats['symbolic_correct'] / stats['total'],
                    'token_accuracy': (stats['token_correct'] / stats['token_total']) / 100 if stats['token_total'] > 0 else 0,
                    'count': stats['total']
                }
        
        return results


class ModelComparator:
    """
    Compare multiple models side-by-side.
    Generates comparison tables and convergence graphs.
    """
    
    def __init__(self, save_dir: str = "/kaggle/working/comparisons/"):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.histories = {}
        self.evaluations = {}
    
    def add_model(self, name: str, history: Dict, evaluation: EvaluationResult,
                  training_time: Optional[float] = None):
        """Add model results for comparison"""
        self.histories[name] = history
        self.evaluations[name] = {
            'result': evaluation,
            'training_time': training_time
        }
    
    def create_comparison_table(self) -> pd.DataFrame:
        """Create side-by-side comparison DataFrame"""
        data = []
        
        for name, eval_data in self.evaluations.items():
            result = eval_data['result']
            row = {
                'Model': name,
                'Token Accuracy': f"{result.token_accuracy:.4f}",
                'Sequence Accuracy': f"{result.sequence_accuracy:.4f}",
                'Symbolic Accuracy': f"{result.symbolic_accuracy:.4f}",
                'BLEU Score': f"{result.bleu_score:.4f}",
                'Perplexity': f"{result.perplexity:.2f}",
                'Training Time (s)': f"{eval_data.get('training_time', 0):.1f}"
            }
            data.append(row)
        
        df = pd.DataFrame(data)
        return df
    
    def plot_convergence(self, metric: str = 'sequence_accuracy', 
                        save_path: Optional[str] = None) -> plt.Figure:
        """
        Plot training convergence for all models.
        
        Args:
            metric: Which metric to plot ('loss', 'token_accuracy', 'sequence_accuracy')
            save_path: Where to save figure
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        for name, history in self.histories.items():
            # Training curves
            if 'train' in history and len(history['train']) > 0:
                train_vals = [h.get(metric, 0) for h in history['train']]
                epochs = range(1, len(train_vals) + 1)
                ax1.plot(epochs, train_vals, label=f'{name} (train)', linestyle='--')
            
            # Validation curves
            if 'val' in history and len(history['val']) > 0:
                val_vals = [h.get(metric, 0) for h in history['val']]
                epochs = range(1, len(val_vals) + 1)
                ax2.plot(epochs, val_vals, label=f'{name} (val)', marker='o')
        
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel(metric.replace('_', ' ').title())
        ax1.set_title('Training Convergence')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel(metric.replace('_', ' ').title())
        ax2.set_title('Validation Convergence')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
        return fig
    
    def plot_category_comparison(self, save_path: Optional[str] = None) -> plt.Figure:
        """Bar chart comparing models by complexity category"""
        categories = set()
        for eval_data in self.evaluations.values():
            categories.update(eval_data['result'].by_complexity.keys())
        
        categories = sorted(categories)
        x = np.arange(len(categories))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        for i, (name, eval_data) in enumerate(self.evaluations.items()):
            result = eval_data['result']
            values = [result.by_complexity.get(cat, {}).get('sequence_accuracy', 0) 
                     for cat in categories]
            offset = width * (i - len(self.evaluations)/2 + 0.5)
            ax.bar(x + offset, values, width, label=name)
        
        ax.set_xlabel('Complexity Category')
        ax.set_ylabel('Sequence Accuracy')
        ax.set_title('Model Comparison by Function Complexity')
        ax.set_xticks(x)
        ax.set_xticklabels(categories, rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
        return fig
    
    def generate_report(self, output_file: str = "comparison_report.md"):
        """Generate comprehensive markdown report"""
        report_path = self.save_dir / output_file
        
        with open(report_path, 'w') as f:
            f.write("# Model Comparison Report\n\n")
            
            # Summary table
            f.write("## Overall Performance\n\n")
            df = self.create_comparison_table()
            f.write(df.to_markdown(index=False))
            f.write("\n\n")
            
            # Detailed breakdown
            f.write("## Performance by Complexity\n\n")
            for name, eval_data in self.evaluations.items():
                f.write(f"### {name}\n\n")
                result = eval_data['result']
                
                f.write("**By Complexity:**\n")
                for cat, metrics in result.by_complexity.items():
                    f.write(f"- {cat}: Seq Acc={metrics['sequence_accuracy']:.3f}, "
                           f"Sym Acc={metrics['symbolic_accuracy']:.3f} "
                           f"(n={metrics['count']})\n")
                
                f.write("\n**By Base Function:**\n")
                for cat, metrics in result.by_base_function.items():
                    if cat.startswith('func_'):
                        func_name = cat[5:]
                        f.write(f"- {func_name}: Seq Acc={metrics['sequence_accuracy']:.3f} "
                               f"(n={metrics['count']})\n")
                f.write("\n")
            
            # Key findings
            f.write("## Key Findings\n\n")
            best_model = max(self.evaluations.items(), 
                           key=lambda x: x[1]['result'].symbolic_accuracy)
            f.write(f"- **Best Overall Model**: {best_model[0]} "
                   f"(Symbolic Accuracy: {best_model[1]['result'].symbolic_accuracy:.3f})\n")
            
            # Identify weaknesses
            for name, eval_data in self.evaluations.items():
                result = eval_data['result']
                weakest = min(result.by_complexity.items(), 
                             key=lambda x: x[1]['sequence_accuracy'])
                f.write(f"- **{name}** struggles most with **{weakest[0]}** functions "
                       f"(accuracy: {weakest[1]['sequence_accuracy']:.3f})\n")
        
        logger.info(f"Generated comparison report at {report_path}")


class AblationStudy:
    """
    Conduct ablation studies to quantify impact of specific components.
    """
    
    def __init__(self, base_config, trainer_class):
        self.base_config = base_config
        self.trainer_class = trainer_class
        self.results = {}
    
    def run_attention_ablation(self, train_data, val_data, test_data):
        """
        Compare LSTM with and without attention mechanism.
        """
        from models import LSTMTaylorModel, ModelConfig
        
        results = {}
        
        # With attention (Bahdanau)
        logger.info("Training LSTM with Bahdanau attention...")
        config_with = self.base_config.copy()
        config_with['attention_type'] = 'bahdanau'
        model_with = LSTMTaylorModel(ModelConfig(**config_with))
        # Train and evaluate...
        
        # Without attention (simple decoder)
        logger.info("Training LSTM without attention...")
        # Would require modifying model to disable attention
        # For now, document the interface
        
        return results
    
    def run_positional_encoding_ablation(self, train_data, val_data, test_data):
        """
        Compare Transformer with sinusoidal vs learned positional encoding.
        """
        from models import TransformerTaylorModel, ModelConfig
        
        results = {}
        
        for pe_type in ['sinusoidal', 'learned']:
            logger.info(f"Training Transformer with {pe_type} PE...")
            config = self.base_config.copy()
            config['positional_encoding'] = pe_type
            model = TransformerTaylorModel(ModelConfig(**config))
            # Train and evaluate...
            results[pe_type] = None  # Store evaluation result
        
        return results


class TaylorEvaluator:
    """
    Main evaluation orchestrator.
    Combines all evaluation components.
    """
    
    def __init__(self, model, tokenizer, device: str = 'auto'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = torch.device(device if device != 'auto' else 
                                   ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.model.to(self.device)
        self.model.eval()
        
        self.verifier = SymbolicVerifier()
        self.visualizer = AttentionVisualizer(tokenizer)
        self.categorized = CategorizedEvaluator(self.verifier)
    
    @torch.no_grad()
    def evaluate(self, test_dataset, batch_size: int = 512, 
                 max_samples: Optional[int] = None) -> EvaluationResult:
        """
        Comprehensive evaluation on test set.
        """
        from torch.utils.data import DataLoader
        
        dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        result = EvaluationResult()
        all_predictions = []
        
        total_tokens = 0
        correct_tokens = 0
        
        pbar = tqdm(dataloader, desc="Evaluating")
        
        for batch_idx, batch in enumerate(pbar):
            if max_samples and batch_idx * batch_size >= max_samples:
                break
            
            # Move to device
            src = batch['encoder_input'].to(self.device)
            tgt = batch['decoder_target'].to(self.device)
            src_mask = batch.get('encoder_mask')
            if src_mask is not None:
                src_mask = src_mask.to(self.device)
            
            # Generate predictions
            max_len = tgt.size(1)
            predictions = self.model.generate(src, src_mask, max_len=max_len)
            
            # Process each sample in batch
            for i in range(src.size(0)):
                # Decode predictions and targets
                pred_ids = predictions[i].cpu().numpy()
                tgt_ids = tgt[i].cpu().numpy()
                
                # Remove padding and special tokens
                pred_str = self.tokenizer.decode(pred_ids, skip_special_tokens=True)
                truth_str = self.tokenizer.decode(tgt_ids, skip_special_tokens=True)
                
                # Token-level accuracy
                min_len = min(len(pred_ids), len(tgt_ids))
                token_matches = (pred_ids[:min_len] == tgt_ids[:min_len]).sum()
                token_total = len([t for t in tgt_ids if t != self.tokenizer.pad_id])
                
                token_acc = token_matches / max(token_total, 1)
                correct_tokens += token_matches
                total_tokens += token_total
                
                # Exact match
                exact_match = pred_str.strip() == truth_str.strip()
                
                # Symbolic verification
                exact_sym, symbolic_match = self.verifier.check_equivalence(pred_str, truth_str)
                
                # Update results
                result.total_samples += 1
                result.correct_predictions += int(exact_match)
                result.symbolic_matches += int(symbolic_match)
                result.equivalence_matches += int(symbolic_match and not exact_match)
                
                # Get sample metadata if available
                sample_meta = {
                    'complexity_class': batch.get('complexity_class', ['unknown']*src.size(0))[i] if isinstance(batch.get('complexity_class'), list) else 'unknown',
                    'base_functions': batch.get('base_functions', [[]]*src.size(0))[i] if isinstance(batch.get('base_functions'), list) else []
                }
                
                # Update categorized results
                self.categorized.update(sample_meta, pred_str, truth_str, 
                                       token_acc, exact_match, symbolic_match)
                
                # Store detailed result
                all_predictions.append({
                    'input': self.tokenizer.decode(src[i].cpu().numpy(), skip_special_tokens=True),
                    'predicted': pred_str,
                    'truth': truth_str,
                    'exact_match': exact_match,
                    'symbolic_match': symbolic_match,
                    'token_accuracy': token_acc,
                    'complexity': sample_meta['complexity_class']
                })
        
        # Calculate final metrics
        result.token_accuracy = correct_tokens / max(total_tokens, 1)
        result.sequence_accuracy = result.correct_predictions / max(result.total_samples, 1)
        result.symbolic_accuracy = result.symbolic_matches / max(result.total_samples, 1)
        result.symbolic_equivalence_rate = result.equivalence_matches / max(result.total_samples, 1)
        result.by_complexity = self.categorized.get_results()
        result.predictions = all_predictions
        
        # Log summary
        logger.info("=" * 60)
        logger.info("EVALUATION RESULTS")
        logger.info("=" * 60)
        logger.info(f"Total Samples: {result.total_samples}")
        logger.info(f"Token Accuracy: {result.token_accuracy:.4f}")
        logger.info(f"Sequence Accuracy: {result.sequence_accuracy:.4f}")
        logger.info(f"Symbolic Accuracy: {result.symbolic_accuracy:.4f}")
        logger.info(f"Equivalent Forms (diff syntax, same math): {result.symbolic_equivalence_rate:.4f}")
        
        logger.info("\nBy Complexity:")
        for cat, metrics in result.by_complexity.items():
            if not cat.startswith('func_'):
                logger.info(f"  {cat:15s}: SeqAcc={metrics['sequence_accuracy']:.3f}, "
                           f"SymAcc={metrics['symbolic_accuracy']:.3f}, "
                           f"n={metrics['count']}")
        
        return result
    
    def visualize_attention(self, test_dataset, num_samples: int = 5):
        """Generate attention visualizations for samples"""
        from torch.utils.data import DataLoader
        
        dataloader = DataLoader(test_dataset, batch_size=8, shuffle=True)
        batch = next(iter(dataloader))
        
        for i in range(min(num_samples, batch['encoder_input'].size(0))):
            self.visualizer.visualize_sample(
                self.model, batch, idx=i,
                save_name=f"attention_sample_{i}.png"
            )
    
    def save_results(self, result: EvaluationResult, path: str):
        """Save detailed results to JSON"""
        with open(path, 'w') as f:
            json.dump(result.to_dict(), f, indent=2)
        logger.info(f"Saved evaluation results to {path}")


# Convenience function for full evaluation pipeline
def run_full_evaluation(model, tokenizer, test_dataset, 
                        history: Optional[Dict] = None,
                        training_time: Optional[float] = None,
                        output_dir: str = "/kaggle/working/reports/") -> Dict[str, Any]:
    """
    Run complete evaluation pipeline and generate all outputs.
    
    Args:
        model: Trained model
        tokenizer: Tokenizer instance
        test_dataset: Test dataset
        history: Training history dict (optional)
        training_time: Total training time in seconds (optional)
        output_dir: Where to save results
    
    Returns:
        Dictionary with all evaluation artifacts
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Initialize evaluator
    evaluator = TaylorEvaluator(model, tokenizer)
    
    # Run evaluation
    result = evaluator.evaluate(test_dataset)
    
    # Save results
    evaluator.save_results(result, "/kaggle/working/metrics.json")
    
    # Visualize attention (if LSTM with attention)
    evaluator.visualize_attention(test_dataset, num_samples=5)
    
    # Generate sample predictions file
    with open("/kaggle/working/FASEROH/working/sample_predictions.json", 'w') as f:
        json.dump(result.predictions[:100], f, indent=2)  # Save first 100
    
    artifacts = {
        'result': result,
        'output_dir': output_dir,
        'history': history,
        'training_time': training_time
    }
    
    return artifacts


if __name__ == "__main__":
    # Example usage demonstration
    print("Taylor Expansion Evaluator")
    print("=" * 60)
    
    # Test symbolic verifier
    verifier = SymbolicVerifier()
    
    test_cases = [
        ("1/2*x", "x/2"),
        ("x**2 + 2*x + 1", "(x+1)**2"),
        ("sin(x)", "sin ( x )"),  # Whitespace difference
        ("x - x**3/6", "x - 1/6*x**3"),
    ]
    
    print("\nSymbolic Verification Tests:")
    for pred, truth in test_cases:
        exact, symbolic = verifier.check_equivalence(pred, truth)
        print(f"  Pred: {pred:20s} | Truth: {truth:20s} | "
              f"Exact: {exact} | Symbolic: {symbolic}")

Writing FASEROH/evaluation/evaluate.py


In [ ]:
%%writefile FASEROH/models/transformer.py

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, Tuple, Union, List, Any
from dataclasses import dataclass
from abc import ABC, abstractmethod
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from FASEROH.training.utils import ModelConfig

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al. 2017)"""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)  # (max_len, 1, d_model)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (seq_len, batch_size, d_model)
        """
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class LearnedPositionalEncoding(nn.Module):
    """Learned positional embeddings (alternative to sinusoidal)"""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.embeddings = nn.Embedding(max_len, d_model)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (seq_len, batch_size, d_model)
        """
        positions = torch.arange(x.size(0), device=x.device).unsqueeze(1)
        pos_emb = self.embeddings(positions)
        return self.dropout(x + pos_emb)

class TransformerEncoderLayer(nn.Module):
    """Transformer Encoder Layer: Multi-Head Attention + FFN"""
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        # Self-attention with residual and norm
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward with residual and norm
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))
        
        return x

class TransformerDecoderLayer(nn.Module):
    """Transformer Decoder Layer: Masked Self-Attn + Cross-Attn + FFN"""
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.masked_self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor,
                tgt_mask: Optional[torch.Tensor] = None,
                src_mask: Optional[torch.Tensor] = None):
        # Masked self-attention (causal)
        attn_out, _ = self.masked_self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Cross-attention to encoder
        attn_out, attn_weights = self.cross_attn(x, encoder_out, encoder_out, key_padding_mask=src_mask)
        x = self.norm2(x + self.dropout(attn_out))
        
        # Feed-forward
        ff_out = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_out))
        
        return x, attn_weights

class TransformerTaylorModel(nn.Module):
    """
    Complete Transformer model for Taylor expansion.
    Supports both sinusoidal and learned positional encodings.
    """
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.model_type = "transformer"
        
        # Embeddings
        self.encoder_embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_idx)
        self.decoder_embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_idx)
        
        # Positional encoding
        if config.positional_encoding == "sinusoidal":
            self.pos_encoder = PositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
            self.pos_decoder = PositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
        else:  # learned
            self.pos_encoder = LearnedPositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
            self.pos_decoder = LearnedPositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
        
        # Encoder stack
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)
        ])
        
        # Decoder stack
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(config.d_model, config.vocab_size)
        
        # Share embeddings if specified
        if config.share_embeddings:
            self.decoder_embedding.weight = self.encoder_embedding.weight
            self.output_proj.weight = self.encoder_embedding.weight  # Weight tying
        
        self._init_weights()
        self.register_buffer('causal_mask', self._generate_causal_mask(config.max_seq_len))
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _generate_causal_mask(self, size: int) -> torch.Tensor:
        """Generate causal (look-ahead) mask using -inf for bulletproof compatibility"""
        mask = torch.triu(torch.ones(size, size), diagonal=1)
        float_mask = mask.masked_fill(mask == 1, float('-inf'))
        return float_mask
    
    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None, return_attention: bool = False) -> torch.Tensor:
        """
        Forward pass with teacher forcing.
        
        Args:
            src: (batch, src_len)
            tgt: (batch, tgt_len)
            src_mask: (batch, src_len) - padding mask for source
            tgt_mask: (batch, tgt_len) - padding mask for target
        """
        batch_size, src_len = src.shape
        tgt_len = tgt.shape[1]

        # Prepend <SOS> (token 2) and drop the last token.
        # This forces the model to predict token t using only tokens up to t-1.
        start_tokens = torch.full((batch_size, 1), 2, dtype=tgt.dtype, device=tgt.device)
        tgt_input = torch.cat([start_tokens, tgt[:, :-1]], dim=1)
        
        # PyTorch expects True for PAD tokens (to ignore). Our loop gives True for VALID tokens.
        src_key_padding_mask = ~src_mask if src_mask is not None else None
        
        # Embed and add positional encoding
        src_emb = self.encoder_embedding(src) * math.sqrt(self.config.d_model)
        src_emb = self.pos_encoder(src_emb.transpose(0, 1)).transpose(0, 1)
        
        tgt_emb = self.decoder_embedding(tgt_input) * math.sqrt(self.config.d_model)
        tgt_emb = self.pos_decoder(tgt_emb.transpose(0, 1)).transpose(0, 1)
        
        # Create causal mask for target
        causal_mask = self.causal_mask[:tgt_len, :tgt_len].to(tgt.device)
        
        # Encode
        encoder_out = src_emb
        for layer in self.encoder_layers:
            encoder_out = layer(encoder_out, src_key_padding_mask)
        
        # Decode
        decoder_out = tgt_emb
        last_attn_weights = None
        for layer in self.decoder_layers:
            decoder_out, last_attn_weights = layer(decoder_out, encoder_out, causal_mask, src_key_padding_mask)
        
        # Project to vocabulary
        logits = self.output_proj(decoder_out)

        if return_attention:
            return logits, last_attn_weights
        
        return logits
    
    def generate(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None,
                 max_len: int = 100, start_token: int = 2, end_token: int = 3,
                 temperature: float = 1.0) -> torch.Tensor:
        
        batch_size = src.size(0)
        device = src.device
        
        src_key_padding_mask = ~src_mask if src_mask is not None else None
        
        # Encode source once
        src_emb = self.encoder_embedding(src) * math.sqrt(self.config.d_model)
        src_emb = self.pos_encoder(src_emb.transpose(0, 1)).transpose(0, 1)
        
        encoder_out = src_emb
        for layer in self.encoder_layers:
            encoder_out = layer(encoder_out, src_key_padding_mask)
            
        # Start with <START> token
        generated = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        
        for i in range(max_len - 1):
            tgt_emb = self.decoder_embedding(generated) * math.sqrt(self.config.d_model)
            tgt_emb = self.pos_decoder(tgt_emb.transpose(0, 1)).transpose(0, 1)
            
            tgt_len = generated.size(1)
            causal_mask = self.causal_mask[:tgt_len, :tgt_len].to(device)
            
            decoder_out = tgt_emb
            for layer in self.decoder_layers:
                decoder_out, _ = layer(decoder_out, encoder_out, causal_mask, src_key_padding_mask)
            
            logits = self.output_proj(decoder_out[:, -1, :])
            
            next_token = logits.argmax(dim=-1, keepdim=True)
            
            pad_idx = self.config.pad_idx if hasattr(self.config, 'pad_idx') else 0
            next_token = next_token.masked_fill(finished.unsqueeze(1), pad_idx)
            
            generated = torch.cat([generated, next_token], dim=1)
            
            finished |= (next_token.squeeze(-1) == end_token)
            if finished.all():
                break
                
        return generated

# Label Smoothing Loss (shared utility)
class LabelSmoothingLoss(nn.Module):
    """
    Label smoothing cross entropy loss.
    Prevents overconfidence and improves generalization.
    """
    def __init__(self, vocab_size: int, smoothing: float = 0.1, pad_idx: int = 0):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing / (vocab_size - 2)  # Exclude pad and true label
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        pred: (batch * seq_len, vocab_size) - logits
        target: (batch * seq_len,) - token indices
        """
        # Ignore padding
        non_pad_mask = (target != self.pad_idx)
        n_tokens = non_pad_mask.sum()
        
        if n_tokens == 0:
            return torch.tensor(0.0, device=pred.device)
        
        pred = pred[non_pad_mask]
        target = target[non_pad_mask]
        
        # Log softmax
        log_probs = F.log_softmax(pred, dim=-1)
        
        # Create smoothed target distribution
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        
        # KL divergence loss
        loss = F.kl_div(log_probs, true_dist, reduction='sum')
        
        return loss / n_tokens


# Visualization utilities
def plot_attention_heatmap(vis_data: Dict, save_path: Optional[str] = None):
    """
    Plot attention heatmap for LSTM model analysis.
    
    Args:
        vis_data: Dict from get_attention_visualization_data containing:
            - attention: numpy array (tgt_len, src_len)
            - source_tokens: list of strings
            - target_tokens: list of strings
    """
    attention = vis_data['attention']
    src_tokens = vis_data['source_tokens']
    tgt_tokens = vis_data['target_tokens']
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention, 
                xticklabels=src_tokens,
                yticklabels=tgt_tokens,
                cmap='viridis',
                cbar_kws={'label': 'Attention Weight'})
    plt.xlabel('Source (Input Function)')
    plt.ylabel('Target (Taylor Expansion)')
    plt.title('Attention Alignment: Function to Taylor Series')
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Example usage
if __name__ == "__main__":
    # Test configuration
    config = ModelConfig(
        vocab_size=100,
        d_model=128,
        max_seq_len=50,
        lstm_hidden=128,
        lstm_layers=2,
        n_layers=2,
        n_heads=4,
        d_ff=512,
        share_embeddings=True,
        label_smoothing=0.1,
        positional_encoding="sinusoidal"
    )
    
    # Test LSTM model
    print("Testing LSTM Model...")
    lstm_model = TaylorModelFactory.create_model("lstm", config)
    
    # Dummy input: batch_size=2, seq_len=10
    src = torch.randint(4, 100, (2, 10))
    tgt = torch.randint(4, 100, (2, 20))
    src_mask = torch.ones(2, 10, dtype=torch.bool)
    
    output = lstm_model(src, tgt, src_mask)
    print(f"LSTM output shape: {output.shape}")  # (2, 19, vocab_size)
    
    # Test generation
    generated = lstm_model.generate(src, src_mask, max_len=20)
    print(f"Generated shape: {generated.shape}")
    
    # Test Transformer model
    print("\nTesting Transformer Model...")
    transformer_model = TaylorModelFactory.create_model("transformer", config)
    
    output = transformer_model(src, tgt, src_mask)
    print(f"Transformer output shape: {output.shape}")
    
    generated = transformer_model.generate(src, src_mask, max_len=20)
    print(f"Generated shape: {generated.shape}")
    
    # Compare parameter counts
    print("\nModel Comparison:")
    comparison = TaylorModelFactory.compare_architectures(vocab_size=1000)
    print(f"LSTM parameters: {comparison['lstm_params']:,}")
    print(f"Transformer parameters: {comparison['transformer_params']:,}")
    
    # Test label smoothing
    criterion = LabelSmoothingLoss(config.vocab_size, config.label_smoothing)
    logits = torch.randn(40, config.vocab_size)
    targets = torch.randint(0, config.vocab_size, (40,))
    loss = criterion(logits, targets)
    print(f"\nLabel smoothing loss: {loss.item():.4f}")

Writing FASEROH/models/transformer.py


In [ ]:
%%writefile FASEROH/models/lstm_seq2seq.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, Tuple, Union, List, Any
from dataclasses import dataclass
from abc import ABC, abstractmethod
import numpy as np
from pathlib import Path

from FASEROH.training.utils import ModelConfig

class BahdanauAttention(nn.Module):
    """
    Bahdanau (Additive) Attention mechanism.
    score(s_t, h_i) = v_a^T * tanh(W_s * s_t + W_h * h_i)
    """
    def __init__(self, hidden_dim: int, encoder_dim: int = None):
        super().__init__()
        encoder_dim = encoder_dim or hidden_dim
        
        self.W_query = nn.Linear(hidden_dim, hidden_dim)
        self.W_key = nn.Linear(encoder_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)
        
    def forward(self, query: torch.Tensor, keys: torch.Tensor, 
                mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            query: (batch, hidden_dim) - decoder hidden state
            keys: (batch, seq_len, encoder_dim) - encoder outputs
            mask: (batch, seq_len) - padding mask (1 for valid, 0 for pad)
        
        Returns:
            context: (batch, encoder_dim) - weighted sum of encoder outputs
            attention_weights: (batch, seq_len) - attention distribution
        """
        query_expanded = self.W_query(query).unsqueeze(1)
        
        # keys: (batch, seq_len, hidden_dim)
        keys_transformed = self.W_key(keys)
        
        # Calculate scores: (batch, seq_len, 1)
        scores = self.V(torch.tanh(query_expanded + keys_transformed)).squeeze(-1)
        
        if mask is not None:
            seq_len = scores.shape[-1]
            if mask.shape[-1] > seq_len:
                mask = mask[..., :seq_len]
                
            # 2. Align dimensions (if scores has a dummy 3D middle dimension)
            if len(scores.shape) == 3 and len(mask.shape) == 2:
                mask = mask.unsqueeze(1)
                
            # 3. Apply the mask safely!
            scores = scores.masked_fill(mask == 0, -1e4)
        
        # Softmax over sequence dimension
        attention_weights = F.softmax(scores, dim=-1)  # (batch, seq_len)
        
        # Weighted sum: (batch, 1, seq_len) @ (batch, seq_len, encoder_dim)
        context = torch.bmm(attention_weights.unsqueeze(1), keys).squeeze(1)
        
        return context, attention_weights

class LuongAttention(nn.Module):
    """
    Luong (Multiplicative) Attention mechanism.
    score(s_t, h_i) = s_t^T * W * h_i
    """
    def __init__(self, hidden_dim: int, encoder_dim: int = None):
        super().__init__()
        encoder_dim = encoder_dim or hidden_dim
        self.W = nn.Linear(encoder_dim, hidden_dim, bias=False)
        
    def forward(self, query: torch.Tensor, keys: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        # query: (batch, hidden_dim)
        # keys: (batch, seq_len, encoder_dim)
        
        # Transform keys: (batch, seq_len, hidden_dim)
        keys_transformed = self.W(keys)
        
        # Calculate scores: (batch, seq_len)
        scores = torch.bmm(keys_transformed, query.unsqueeze(-1)).squeeze(-1)
        
        if mask is not None:
            # 1. PyTorch's LSTM optimizes by dropping trailing pads,
            #    shrinking the sequence length (e.g., from 50 down to 31).
            #    We must slice the mask to perfectly match this new optimized length!
            seq_len = scores.shape[-1]
            if mask.shape[-1] > seq_len:
                mask = mask[..., :seq_len]
                
            # 2. Align dimensions (if scores has a dummy 3D middle dimension)
            if len(scores.shape) == 3 and len(mask.shape) == 2:
                mask = mask.unsqueeze(1)
                
            # 3. Apply the mask safely!
            scores = scores.masked_fill(mask == 0, -1e4)
        
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attention_weights.unsqueeze(1), keys).squeeze(1)
        
        return context, attention_weights

class LSTMEncoder(nn.Module):
    """Bidirectional LSTM Encoder with optional projection"""
    def __init__(self, vocab_size: int, d_model: int, hidden_dim: int, 
                 n_layers: int = 2, dropout: float = 0.1, 
                 bidirectional: bool = True, pad_idx: int = 0):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            d_model, hidden_dim, n_layers, 
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=bidirectional,
            batch_first=True
        )
        
        # If bidirectional, project to hidden_dim for decoder init
        self.bidirectional = bidirectional
        if bidirectional:
            self.hidden_proj = nn.Linear(hidden_dim * 2, hidden_dim)
            self.cell_proj = nn.Linear(hidden_dim * 2, hidden_dim)
            
    def forward(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        """
        Args:
            src: (batch, seq_len) - token IDs
            src_mask: (batch, seq_len) - padding mask
        
        Returns:
            outputs: (batch, seq_len, hidden_dim * num_directions)
            hidden: (num_layers, batch, hidden_dim)
            cell: (num_layers, batch, hidden_dim)
        """
        embedded = self.embedding(src)  # (batch, seq_len, d_model)
        
        # Pack sequence for efficiency if mask provided
        if src_mask is not None:
            lengths = src_mask.sum(dim=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(
                embedded, lengths, batch_first=True, enforce_sorted=False
            )
            outputs, (hidden, cell) = self.lstm(packed)
            outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        else:
            outputs, (hidden, cell) = self.lstm(embedded)
        
        # If bidirectional, concatenate directions and project for decoder init
        if self.bidirectional:
            # hidden: (num_layers * 2, batch, hidden_dim)
            batch_size = hidden.size(1)
            
            # Reshape and project
            # Separate forward and backward, then project
            hidden = hidden.view(self.lstm.num_layers, 2, batch_size, self.lstm.hidden_size)
            cell = cell.view(self.lstm.num_layers, 2, batch_size, self.lstm.hidden_size)
            
            # Concatenate forward and backward
            hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=-1)
            cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=-1)
            
            # Project back to hidden_dim
            hidden = self.hidden_proj(hidden)
            cell = self.cell_proj(cell)
            
            # outputs: project from hidden*2 to hidden
            outputs = self.hidden_proj(outputs)
        
        return outputs, hidden, cell

class LSTMDecoder(nn.Module):
    """LSTM Decoder with Attention mechanism"""
    def __init__(self, vocab_size: int, d_model: int, hidden_dim: int,
                 encoder_dim: int, n_layers: int = 2, 
                 attention_type: str = "bahdanau", dropout: float = 0.1,
                 pad_idx: int = 0):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # LSTM input is embedding + context vector
        self.lstm = nn.LSTM(
            d_model + encoder_dim,  # Concatenate embedding and context
            hidden_dim, 
            n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        # Attention mechanism
        if attention_type == "bahdanau":
            self.attention = BahdanauAttention(hidden_dim, encoder_dim)
        else:  # luong
            self.attention = LuongAttention(hidden_dim, encoder_dim)
        
        # Output projection: LSTM output + context -> vocab
        self.output_proj = nn.Linear(hidden_dim + encoder_dim, vocab_size)
        
        # Store attention weights for visualization
        self.attention_weights_history: List[torch.Tensor] = []
        self.store_attention = False
        
    def forward(self, input_token: torch.Tensor, hidden: torch.Tensor, 
                cell: torch.Tensor, encoder_outputs: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Single step forward (for autoregressive generation)
        
        Args:
            input_token: (batch, 1) - single token ID
            hidden: (n_layers, batch, hidden_dim)
            cell: (n_layers, batch, hidden_dim)
            encoder_outputs: (batch, src_len, encoder_dim)
            src_mask: (batch, src_len)
        
        Returns:
            output: (batch, vocab_size) - logits
            hidden: updated hidden state
            cell: updated cell state
        """
        # input_token: (batch, 1) -> (batch, 1, d_model)
        embedded = self.embedding(input_token)
        
        # Use last layer hidden state for attention query
        query = hidden[-1]  # (batch, hidden_dim)
        
        # Calculate attention
        context, attn_weights = self.attention(query, encoder_outputs, src_mask)
        
        # Store attention weights if visualization enabled
        if self.store_attention:
            self.attention_weights_history.append(attn_weights.detach().cpu())
        
        # Concatenate embedding and context
        lstm_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)
        
        # LSTM step
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Prepare for output projection: concat output and context
        output = output.squeeze(1)  # (batch, hidden_dim)
        output = torch.cat([output, context], dim=-1)
        
        # Project to vocabulary
        logits = self.output_proj(output)
        
        return logits, hidden, cell
    
    def enable_attention_storage(self, enabled: bool = True):
        """Enable/disable attention weight storage for visualization"""
        self.store_attention = enabled
        if not enabled:
            self.attention_weights_history.clear()
    
    def get_attention_matrix(self) -> Optional[np.ndarray]:
        """Get attention weights as numpy array (time_steps, src_len)"""
        if not self.attention_weights_history:
            return None
        
        # Stack along time dimension
        attn = torch.stack(self.attention_weights_history, dim=1)  # (batch, tgt_len, src_len)
        return attn.numpy()


class LSTMTaylorModel(nn.Module):
    """
    Complete LSTM Seq2Seq model with Attention for Taylor expansion.
    Includes hooks for attention visualization.
    """
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.model_type = "lstm"
        
        # Encoder
        encoder_dim = config.lstm_hidden * (2 if config.lstm_bidirectional else 1)
        self.encoder = LSTMEncoder(
            vocab_size=config.vocab_size,
            d_model=config.d_model,
            hidden_dim=config.lstm_hidden,
            n_layers=config.lstm_layers,
            dropout=config.dropout,
            bidirectional=config.lstm_bidirectional,
            pad_idx=config.pad_idx
        )
        
        # Decoder
        self.decoder = LSTMDecoder(
            vocab_size=config.vocab_size,
            d_model=config.d_model,
            hidden_dim=config.lstm_hidden,
            encoder_dim=config.lstm_hidden,  # After projection
            n_layers=config.lstm_layers,
            attention_type=config.attention_type,
            dropout=config.dropout,
            pad_idx=config.pad_idx
        )
        
        # Share embeddings if specified
        if config.share_embeddings:
            self.decoder.embedding.weight = self.encoder.embedding.weight
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Teacher forcing training forward pass.
        
        Args:
            src: (batch, src_len) - input function tokens
            tgt: (batch, tgt_len) - target Taylor expansion tokens
            src_mask: (batch, src_len) - padding mask for source
            tgt_mask: (batch, tgt_len) - padding mask for target
        
        Returns:
            outputs: (batch, tgt_len, vocab_size) - logits
        """
        batch_size, tgt_len = tgt.size()
        
        # Encode
        encoder_outputs, hidden, cell = self.encoder(src, src_mask)
        
        # Prepare decoder inputs (shift right, start with <START>)
        # Assuming tgt is already shifted appropriately by data pipeline
        decoder_input = tgt[:, :-1]  # Exclude last token
        
        # Storage for outputs
        outputs = torch.zeros(batch_size, tgt_len - 1, self.config.vocab_size,
                            device=src.device)
        
        # Decode step by step
        for t in range(tgt_len - 1):
            input_token = decoder_input[:, t].unsqueeze(1)
            logits, hidden, cell = self.decoder(
                input_token, hidden, cell, encoder_outputs, src_mask
            )
            outputs[:, t, :] = logits
        
        return outputs

    @torch.no_grad()
    def generate(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None,
                 max_len: int = 100, start_token: int = 2, end_token: int = 3) -> torch.Tensor:
        """
        Autoregressive Greedy Generation.
        For mathematical tasks, we always use argmax (greedy) decoding 
        to prevent hallucinations and CUDA sampling deadlocks.
        """
        batch_size = src.size(0)
        device = src.device
        
        # Encode
        encoder_outputs, hidden, cell = self.encoder(src, src_mask)
        
        # Start with <START> token
        input_token = torch.full((batch_size, 1), start_token, 
                                dtype=torch.long, device=device)
        generated = [input_token]
        
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        
        for _ in range(max_len - 1):
            logits, hidden, cell = self.decoder(
                input_token, hidden, cell, encoder_outputs, src_mask
            )
            
            # --- THE FIX: Pure Greedy Decoding (Argmax) ---
            # Grabs the most likely token. No multinomial sampling, no NaN deadlocks.
            next_token = logits.argmax(dim=-1, keepdim=True)
            
            generated.append(next_token)
            
            # Check for end tokens
            finished |= (next_token.squeeze(-1) == end_token)
            if finished.all():
                break
            
            input_token = next_token
        
        return torch.cat(generated, dim=1)
        
    def get_attention_visualization_data(self, src: torch.Tensor, 
                                        tgt: torch.Tensor,
                                        src_tokens: List[str],
                                        tgt_tokens: List[str]) -> Dict:
        """
        Generate attention heatmap data for visualization.
        
        Returns dict with attention matrix and token labels.
        """
        self.decoder.enable_attention_storage(True)
        
        with torch.no_grad():
            _ = self.forward(src.unsqueeze(0), tgt.unsqueeze(0))
        
        attn_matrix = self.decoder.get_attention_matrix()[0]  # First batch item
        
        self.decoder.enable_attention_storage(False)
        
        return {
            'attention': attn_matrix,
            'source_tokens': src_tokens,
            'target_tokens': tgt_tokens[:attn_matrix.shape[0]]
        }

# Example usage
if __name__ == "__main__":
    # Test configuration
    config = ModelConfig(
        vocab_size=100,
        d_model=128,
        max_seq_len=50,
        lstm_hidden=128,
        lstm_layers=2,
        n_layers=2,
        n_heads=4,
        d_ff=512,
        share_embeddings=True,
        label_smoothing=0.1,
        positional_encoding="sinusoidal"
    )
    
    # Test LSTM model
    print("Testing LSTM Model...")
    lstm_model = TaylorModelFactory.create_model("lstm", config)
    
    # Dummy input: batch_size=2, seq_len=10
    src = torch.randint(4, 100, (2, 10))
    tgt = torch.randint(4, 100, (2, 20))
    src_mask = torch.ones(2, 10, dtype=torch.bool)
    
    output = lstm_model(src, tgt, src_mask)
    print(f"LSTM output shape: {output.shape}")  # (2, 19, vocab_size)
    
    # Test generation
    generated = lstm_model.generate(src, src_mask, max_len=20)
    print(f"Generated shape: {generated.shape}")
    
    # Test Transformer model
    print("\nTesting Transformer Model...")
    transformer_model = TaylorModelFactory.create_model("transformer", config)
    
    output = transformer_model(src, tgt, src_mask)
    print(f"Transformer output shape: {output.shape}")
    
    generated = transformer_model.generate(src, src_mask, max_len=20)
    print(f"Generated shape: {generated.shape}")
    
    # Compare parameter counts
    print("\nModel Comparison:")
    comparison = TaylorModelFactory.compare_architectures(vocab_size=1000)
    print(f"LSTM parameters: {comparison['lstm_params']:,}")
    print(f"Transformer parameters: {comparison['transformer_params']:,}")
    
    # Test label smoothing
    criterion = LabelSmoothingLoss(config.vocab_size, config.label_smoothing)
    logits = torch.randn(40, config.vocab_size)
    targets = torch.randint(0, config.vocab_size, (40,))
    loss = criterion(logits, targets)
    print(f"\nLabel smoothing loss: {loss.item():.4f}")

Writing FASEROH/models/lstm_seq2seq.py


In [ ]:
%%writefile FASEROH/training/utils.py

from dataclasses import dataclass
import torch.nn.functional as F
import torch
import torch.nn as nn
from typing import Dict, Optional, Tuple, Union, List, Any
import matplotlib.pyplot as plt
import seaborn as sns

@dataclass
class ModelConfig:
    """Unified configuration for both LSTM and Transformer models"""
    vocab_size: int = 1000
    d_model: int = 256          # Embedding dimension
    max_seq_len: int = 100      # Maximum sequence length
    
    # LSTM specific
    lstm_hidden: int = 256
    lstm_layers: int = 2
    lstm_bidirectional: bool = True
    attention_type: str = "bahdanau"  # "bahdanau" or "luong"
    
    # Transformer specific
    n_layers: int = 4           # Encoder/decoder layers
    n_heads: int = 8            # Attention heads
    d_ff: int = 1024            # Feed-forward dimension
    dropout: float = 0.1
    positional_encoding: str = "sinusoidal"  # "sinusoidal" or "learned"
    
    # Shared settings
    share_embeddings: bool = True
    label_smoothing: float = 0.1
    pad_idx: int = 0
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            'vocab_size': self.vocab_size,
            'd_model': self.d_model,
            'max_seq_len': self.max_seq_len,
            'lstm_hidden': self.lstm_hidden,
            'lstm_layers': self.lstm_layers,
            'lstm_bidirectional': self.lstm_bidirectional,
            'attention_type': self.attention_type,
            'n_layers': self.n_layers,
            'n_heads': self.n_heads,
            'd_ff': self.d_ff,
            'dropout': self.dropout,
            'positional_encoding': self.positional_encoding,
            'share_embeddings': self.share_embeddings,
            'label_smoothing': self.label_smoothing,
            'pad_idx': self.pad_idx
        }

class TaylorModelFactory:
    """Factory to create models with identical interface"""
    
    @staticmethod
    # def create_model(model_type: str, config: ModelConfig) -> Union[LSTMTaylorModel, TransformerTaylorModel]:
    def create_model(model_type: str, config: ModelConfig):

        """
        Create model instance based on type.
        
        Args:
            model_type: "lstm" or "transformer"
            config: Model configuration
        
        Returns:
            Model instance with forward/generge interface
        """
        if model_type.lower() == "lstm":
            from FASEROH.models.lstm_seq2seq import LSTMTaylorModel
            return LSTMTaylorModel(config)
        elif model_type.lower() == "transformer":
            from FASEROH.models.transformer import TransformerTaylorModel
            return TransformerTaylorModel(config)
        else:
            raise ValueError(f"Unknown model type: {model_type}")
    
    @staticmethod
    def compare_architectures(vocab_size: int = 1000) -> Dict[str, int]:
        """
        Print parameter counts for both architectures.
        Useful for model comparison in reports.
        """
        config = ModelConfig(vocab_size=vocab_size)
        
        lstm_model = TaylorModelFactory.create_model("lstm", config)
        transformer_model = TaylorModelFactory.create_model("transformer", config)
        
        lstm_params = sum(p.numel() for p in lstm_model.parameters())
        trans_params = sum(p.numel() for p in transformer_model.parameters())
        
        return {
            'lstm_params': lstm_params,
            'transformer_params': trans_params,
            'ratio': trans_params / lstm_params if lstm_params > 0 else 0
        }

# Label Smoothing Loss (shared utility)
class LabelSmoothingLoss(nn.Module):
    """
    Label smoothing cross entropy loss.
    Prevents overconfidence and improves generalization.
    """
    def __init__(self, vocab_size: int, smoothing: float = 0.1, pad_idx: int = 0):
        super().__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing / (vocab_size - 2)  # Exclude pad and true label
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        pred: (batch * seq_len, vocab_size) - logits
        target: (batch * seq_len,) - token indices
        """
        # Ignore padding
        non_pad_mask = (target != self.pad_idx)
        n_tokens = non_pad_mask.sum()
        
        if n_tokens == 0:
            return torch.tensor(0.0, device=pred.device)
        
        pred = pred[non_pad_mask]
        target = target[non_pad_mask]
        
        # Log softmax
        log_probs = F.log_softmax(pred, dim=-1)
        
        # Create smoothed target distribution
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        
        # KL divergence loss
        loss = F.kl_div(log_probs, true_dist, reduction='sum')
        
        return loss / n_tokens

# Visualization utilities
def plot_attention_heatmap(vis_data: Dict, save_path: Optional[str] = None):
    """
    Plot attention heatmap for LSTM model analysis.
    
    Args:
        vis_data: Dict from get_attention_visualization_data containing:
            - attention: numpy array (tgt_len, src_len)
            - source_tokens: list of strings
            - target_tokens: list of strings
    """
    attention = vis_data['attention']
    src_tokens = vis_data['source_tokens']
    tgt_tokens = vis_data['target_tokens']
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention, 
                xticklabels=src_tokens,
                yticklabels=tgt_tokens,
                cmap='viridis',
                cbar_kws={'label': 'Attention Weight'})
    plt.xlabel('Source (Input Function)')
    plt.ylabel('Target (Taylor Expansion)')
    plt.title('Attention Alignment: Function to Taylor Series')
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import numpy as np
import json
import time
import math
from typing import Dict, List, Optional, Tuple, Union, Any
from dataclasses import dataclass, field
from pathlib import Path
import logging
from collections import defaultdict
import copy
import warnings

import sys
import os
sys.path.append("/teamspace/studios/this_studio/")

# For BLEU score calculation
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    NLTK_AVAILABLE = True
except ImportError:
    NLTK_AVAILABLE = False
    warnings.warn("NLTK not available. BLEU scores will not be calculated.")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


@dataclass
class TrainingConfig:
    """Comprehensive training configuration"""
    # Optimization
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    batch_size: int = 512
    epochs: int = 100
    gradient_clip_norm: float = 1.0
    label_smoothing: float = 0.1
    
    # Learning rate scheduling
    lr_scheduler: str = "reduce_on_plateau"  # "reduce_on_plateau", "cosine_warm_restarts", "none"
    lr_patience: int = 5  # for ReduceLROnPlateau
    lr_factor: float = 0.5  # for ReduceLROnPlateau
    T_0: int = 10  # for CosineAnnealingWarmRestarts (first restart)
    T_mult: int = 2  # for CosineAnnealingWarmRestarts (multiplication factor)
    
    # Early stopping
    early_stopping_patience: int = 10
    monitor_metric: str = "loss"  # Checkpoint based on this
    
    # Curriculum learning
    use_curriculum: bool = True
    curriculum_epochs: List[int] = field(default_factory=lambda: [10, 20, 30, 40])
    # Each entry is epoch to switch to next complexity level

    # Checkpointing
    checkpoint_dir: str = "/kaggle/working/checkpoints/"
    save_best_only: bool = True
    
    # Validation
    validation_split: float = 0.2
    validate_every: int = 1  # Validate every N epochs
    
    # Teacher forcing
    teacher_forcing_ratio: float = 1.0  # Always 1.0 for training (handled by model)
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            'learning_rate': self.learning_rate,
            'weight_decay': self.weight_decay,
            'batch_size': self.batch_size,
            'epochs': self.epochs,
            'gradient_clip_norm': self.gradient_clip_norm,
            'lr_scheduler': self.lr_scheduler,
            'early_stopping_patience': self.early_stopping_patience,
            'monitor_metric': self.monitor_metric,
            'use_curriculum': self.use_curriculum,
        }


class MetricsCalculator:
    """Calculate and track all training metrics"""
    
    def __init__(self, pad_idx: int = 0, vocab=None):
        self.pad_idx = pad_idx
        self.vocab = vocab  # For BLEU score detokenization if needed
        self.reset()
        
        if NLTK_AVAILABLE:
            self.smoothing = SmoothingFunction().method1
    
    def reset(self):
        self.total_tokens = 0
        self.correct_tokens = 0
        self.total_sequences = 0
        self.correct_sequences = 0
        self.total_loss = 0.0
        self.batch_count = 0
        self.all_predictions = []
        self.all_targets = []
    
    def update(self, predictions: torch.Tensor, targets: torch.Tensor, 
               loss: float, mask: Optional[torch.Tensor] = None):
        """
        Update metrics with batch results.
        
        Args:
            predictions: (batch, seq_len) predicted token IDs
            targets: (batch, seq_len) target token IDs
            loss: batch loss value
            mask: (batch, seq_len) valid token mask
        """
        self.total_loss += loss
        self.batch_count += 1
        
        if mask is None:
            mask = (targets != self.pad_idx)
        
        # Token accuracy (excluding padding)
        valid_tokens = mask.sum().item()
        correct = ((predictions == targets) & mask).sum().item()
        self.correct_tokens += correct
        self.total_tokens += valid_tokens
        
        # Sequence accuracy (exact match, excluding padding)
        # Check if all valid tokens match
        matches = (predictions == targets) | ~mask
        seq_correct = matches.all(dim=1).sum().item()
        self.correct_sequences += seq_correct
        self.total_sequences += predictions.size(0)
        
        # Store for BLEU calculation (do it at end for efficiency)
        self.all_predictions.extend(predictions.cpu().numpy())
        self.all_targets.extend(targets.cpu().numpy())
    
    def compute(self) -> Dict[str, float]:
        """Compute final metrics"""
        metrics = {}
        
        # Loss
        metrics['loss'] = self.total_loss / max(self.batch_count, 1)
        
        # Token accuracy
        metrics['token_accuracy'] = self.correct_tokens / max(self.total_tokens, 1)
        
        # Sequence accuracy
        metrics['sequence_accuracy'] = self.correct_sequences / max(self.total_sequences, 1)
        
        # Perplexity
        try:
            metrics['perplexity'] = math.exp(metrics['loss'])
        except OverflowError:
            metrics['perplexity'] = float('inf')
        
        # BLEU score (if available)
        if NLTK_AVAILABLE and self.vocab is not None:
            bleu_scores = []
            for pred, tgt in zip(self.all_predictions, self.all_targets):
                # Remove padding and convert to lists
                pred_tokens = [int(p) for p in pred if p != self.pad_idx]
                tgt_tokens = [int(t) for t in tgt if t != self.pad_idx]
                
                if len(pred_tokens) > 0 and len(tgt_tokens) > 0:
                    score = sentence_bleu(
                        [tgt_tokens], 
                        pred_tokens,
                        smoothing_function=self.smoothing
                    )
                    bleu_scores.append(score)
            
            metrics['bleu'] = np.mean(bleu_scores) if bleu_scores else 0.0
        else:
            metrics['bleu'] = 0.0
        
        return metrics


class CurriculumScheduler:
    """
    Curriculum learning scheduler.
    Progressively increases difficulty based on epochs.
    """
    
    def __init__(self, schedule: Optional[List[int]] = None):
        """
        Args:
            schedule: List of epoch numbers where difficulty increases.
                     At epoch schedule[0], allow level 1, etc.
        """
        self.schedule = schedule or [10, 20, 30, 40]
        self.current_level = 0
        self.complexity_levels = ['polynomial', 'basic', 'binary', 'composition', 'complex']
        
    def get_allowed_complexities(self, epoch: int) -> List[str]:
        """Get list of allowed complexity classes for current epoch"""
        level = 0
        for threshold in self.schedule:
            if epoch >= threshold:
                level += 1
        
        level = min(level, len(self.complexity_levels) - 1)
        self.current_level = level
        
        # Return all complexities up to current level
        return self.complexity_levels[:level + 1]
    
    def filter_dataset(self, dataset, epoch: int):
        """
        Filter dataset to include only samples up to current complexity level.
        Assumes dataset items have 'complexity_class' field.
        """
        allowed = self.get_allowed_complexities(epoch)
        indices = []
        
        for i, item in enumerate(dataset):
            # Handle both dict-style and object-style datasets
            if isinstance(item, dict):
                complexity = item.get('complexity_class', 'basic')
            else:
                complexity = getattr(item, 'complexity_class', 'basic')
            
            if complexity in allowed:
                indices.append(i)
        
        return indices
    
    def __repr__(self):
        return f"CurriculumScheduler(current_level={self.current_level}, allowed={self.get_allowed_complexities(0)})"


class EarlyStopping:
    """Early stopping based on monitored metric (higher is better for accuracy)"""
    
    def __init__(self, patience: int = 10, min_delta: float = 0.0, 
                 mode: str = 'max', verbose: bool = True):
        """
        Args:
            patience: Number of epochs to wait before stopping
            min_delta: Minimum change to qualify as improvement
            mode: 'max' for accuracy, 'min' for loss
            verbose: Whether to print messages
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_value = None
        self.early_stop = False
        
    def __call__(self, metric_value: float) -> bool:
        if self.best_value is None:
            self.best_value = metric_value
            return False
        
        if self.mode == 'max':
            improved = metric_value > self.best_value + self.min_delta
        else:
            improved = metric_value < self.best_value - self.min_delta
        
        if improved:
            self.best_value = metric_value
            self.counter = 0
            if self.verbose:
                logger.info(f"Metric improved to {metric_value:.4f}")
        else:
            self.counter += 1
            if self.verbose:
                logger.info(f"No improvement for {self.counter} epochs. Best: {self.best_value:.4f}")
            
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    logger.info("Early stopping triggered!")
        
        return self.early_stop


class CheckpointManager:
    """Manage model checkpointing based on monitored metric"""
    
    def __init__(self, checkpoint_dir: str, monitor: str = 'val_seq_accuracy', 
                 mode: str = 'max', save_best_only: bool = True):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.monitor = monitor
        self.mode = mode
        self.save_best_only = save_best_only
        self.best_value = float('-inf') if mode == 'max' else float('inf')
        self.best_path = None
        
    def save_checkpoint(self, model, optimizer, scheduler, epoch: int, 
                       metrics: Dict[str, float], is_best: bool = False):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'metrics': metrics,
            'best_value': self.best_value
        }
        
        # Save latest
        latest_path = self.checkpoint_dir / 'checkpoint_latest.pt'
        torch.save(checkpoint, latest_path)
        
        # Save best if improved
        if is_best:
            self.best_path = self.checkpoint_dir / 'checkpoint_best.pt'
            torch.save(checkpoint, self.best_path)
            logger.info(f"Saved best checkpoint with {self.monitor}={metrics[self.monitor]:.4f}")
        
        # Save periodic (every 10 epochs)
        if epoch % 10 == 0:
            periodic_path = self.checkpoint_dir / f'checkpoint_epoch_{epoch}.pt'
            torch.save(checkpoint, periodic_path)
    
    def check_improvement(self, metric_value: float) -> bool:
        """Check if metric improved"""
        if self.mode == 'max':
            improved = metric_value > self.best_value
        else:
            improved = metric_value < self.best_value
        
        if improved:
            self.best_value = metric_value
        
        return improved
    
    def load_checkpoint(self, model, optimizer=None, scheduler=None, 
                       checkpoint_path: Optional[str] = None) -> Dict:
        """Load checkpoint"""
        if checkpoint_path is None:
            checkpoint_path = self.best_path or (self.checkpoint_dir / 'checkpoint_latest.pt')
        
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if optimizer and checkpoint['optimizer_state_dict']:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        if scheduler and checkpoint.get('scheduler_state_dict'):
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        logger.info(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
        return checkpoint


class Trainer:
    """
    Complete training pipeline with curriculum learning, LR scheduling,
    gradient clipping, and comprehensive metrics.
    """
    
    def __init__(self, model, config: TrainingConfig, device: str = 'auto'):
        self.model = model
        self.config = config
        self.device = self._get_device(device)
        self.model.to(self.device)
        self.scaler = torch.cuda.amp.GradScaler()
        
        # Optimizer
        self.optimizer = optim.Adam(
            model.parameters(), 
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
        
        # LR Scheduler
        self.scheduler = self._init_scheduler()
        
        # Loss function with label smoothing
        self.criterion = self._init_criterion()
        
        # Curriculum learning
        self.curriculum = CurriculumScheduler(config.curriculum_epochs) if config.use_curriculum else None
        
        # Early stopping (monitor sequence accuracy, higher is better)
        mode = 'max' if 'accuracy' in config.monitor_metric else 'min'
        self.early_stopping = EarlyStopping(
            patience=config.early_stopping_patience,
            mode=mode
        )
        
        # Checkpoint manager
        self.checkpoint_manager = CheckpointManager(
            config.checkpoint_dir,
            monitor=config.monitor_metric,
            mode=mode,
            save_best_only=config.save_best_only
        )
        
        # Metrics tracking
        self.train_metrics_history = []
        self.val_metrics_history = []
        self.current_epoch = 0
        
    def _get_device(self, device: str) -> torch.device:
        if device == 'auto':
            return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        return torch.device(device)
    
    def _init_scheduler(self):
        """Initialize learning rate scheduler"""
        if self.config.lr_scheduler == 'reduce_on_plateau':
            return optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer,
                mode='max' if 'accuracy' in self.config.monitor_metric else 'min',
                factor=self.config.lr_factor,
                patience=self.config.lr_patience
                # verbose=True
            )
        elif self.config.lr_scheduler == 'cosine_warm_restarts':
            return optim.lr_scheduler.CosineAnnealingWarmRestarts(
                self.optimizer,
                T_0=self.config.T_0,
                T_mult=self.config.T_mult
            )
        else:
            return None
    
    def _init_criterion(self):
        """Initialize loss function"""
        # Use label smoothing if specified
        if self.config.label_smoothing > 0:
            from FASEROH.training.utils import LabelSmoothingLoss  # Assuming from previous phase
            return LabelSmoothingLoss(
                vocab_size=self.model.config.vocab_size,
                smoothing=self.config.label_smoothing,
                pad_idx=self.model.config.pad_idx
            )
        else:
            return nn.CrossEntropyLoss(ignore_index=self.model.config.pad_idx)

    def _apply_curriculum(self, train_dataset, epoch: int):
        """Apply curriculum learning by filtering dataset"""
        if self.curriculum is None:
            return train_dataset
        
        allowed = self.curriculum.get_allowed_complexities(epoch)
        logger.info(f"Epoch {epoch}: Curriculum level allows {allowed}")
        
        # Filter indices
        indices = self.curriculum.filter_dataset(train_dataset, epoch)
        
        if len(indices) == 0:
            logger.warning("No samples match curriculum criteria, using full dataset")
            return train_dataset
        
        logger.info(f"Curriculum filtered dataset: {len(indices)}/{len(train_dataset)} samples")
        return torch.utils.data.Subset(train_dataset, indices)

    def _train_epoch(self, dataloader: DataLoader, epoch: int) -> Dict[str, float]:
        """Train for one epoch"""
        self.model.train()
        metrics_calc = MetricsCalculator(pad_idx=self.model.config.pad_idx)
        
        epoch_loss = 0.0
        
        for batch_idx, batch in enumerate(dataloader):
            # Move to device
            src = batch['encoder_input'].to(self.device)
            tgt = batch['decoder_target'].to(self.device)
            src_mask = (src != self.model.config.pad_idx).to(self.device)
            
            self.optimizer.zero_grad()
            
            # 1. AMP FORWARD PASS
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = self.model(src, tgt, src_mask)

                seq_len = logits.size(1)
                targets_aligned = tgt[:, -seq_len:]

                batch_size, _, vocab_size = logits.shape
                logits_flat = logits.reshape(-1, vocab_size)
                targets_flat = targets_aligned.reshape(-1)
                
                loss = self.criterion(logits_flat, targets_flat)
            
            # 2. AMP BACKWARD PASS
            self.scaler.scale(loss).backward()
            
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            # 3. AMP OPTIMIZER STEP
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            # Update metrics
            with torch.no_grad():
                predictions = logits.argmax(dim=-1)
                metrics_calc.update(predictions, targets_aligned, loss.item())
            
            epoch_loss += loss.item()
            
            if batch_idx % 100 == 0:
                logger.info(f"Epoch {epoch} [{batch_idx}/{len(dataloader)}] Loss: {loss.item():.4f}")
        
        return metrics_calc.compute()
    
    @torch.no_grad()
    def _validate(self, dataloader: DataLoader) -> Dict[str, float]:
        """Validate model"""
        self.model.eval()
        metrics_calc = MetricsCalculator(pad_idx=self.model.config.pad_idx)
        
        for batch in dataloader:
            # batch = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            src = batch['encoder_input'].to(self.device)
            tgt = batch['decoder_target'].to(self.device)
            # src_mask = batch.get('encoder_mask', None)
            src_mask = (src != self.model.config.pad_idx).to(self.device)
            if src_mask is not None:
                src_mask = src_mask.to(self.device)
            
            # Generate predictions (no teacher forcing)
            max_len = tgt.size(1)
            generated = self.model.generate(src, src_mask, max_len=max_len)
            
            # Pad or truncate to match target length for metric calculation
            if generated.size(1) < max_len:
                padding = torch.full(
                    (generated.size(0), max_len - generated.size(1)), 
                    self.model.config.pad_idx,
                    device=self.device
                )
                generated = torch.cat([generated, padding], dim=1)
            else:
                generated = generated[:, :max_len]
            
            # Calculate validation loss (with teacher forcing for consistency)
            logits = self.model(src, tgt, src_mask)

            seq_len = logits.size(1)
            targets_aligned = tgt[:, -seq_len:]
            
            batch_size, _, vocab_size = logits.shape
            logits_flat = logits.reshape(-1, vocab_size)
            targets_flat = targets_aligned.reshape(-1)
            
            loss = self.criterion(logits_flat, targets_flat).item()
            
            metrics_calc.update(generated, tgt, loss)
        
        return metrics_calc.compute()
    
    def fit(self, train_dataset, val_dataset=None):
        """
        Main training loop.
        
        Args:
            train_dataset: Training dataset
            val_dataset: Validation dataset (if None, uses subset of train)
        """
        logger.info("Starting training...")
        logger.info(f"Device: {self.device}")
        logger.info(f"Model parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        
        # Create validation split if not provided
        if val_dataset is None:
            val_size = int(len(train_dataset) * self.config.validation_split)
            train_size = len(train_dataset) - val_size
            train_dataset, val_dataset = torch.utils.data.random_split(
                train_dataset, [train_size, val_size]
            )
            logger.info(f"Split train into {train_size} train / {val_size} val")
        
        best_metric_value = float('-inf') if 'accuracy' in self.config.monitor_metric else float('inf')
        
        if self.config.lr_scheduler == "onecycle":
            # Estimate total steps based on the full dataset
            dummy_loader = DataLoader(train_dataset, batch_size=self.config.batch_size)
            total_steps = len(dummy_loader) * self.config.epochs
            
            self.scheduler = optim.lr_scheduler.OneCycleLR(
                self.optimizer,
                max_lr=self.config.learning_rate,
                total_steps=total_steps,
                pct_start=0.1, # 10% warmup
                anneal_strategy='cos'
            )
            logger.info(f"Initialized OneCycleLR Warmup for {total_steps} total steps.")
        
        for epoch in range(self.current_epoch, self.config.epochs):
            self.current_epoch = epoch
            epoch_start_time = time.time()
            
            # Apply curriculum learning
            if self.curriculum:
                current_train_data = self._apply_curriculum(train_dataset, epoch)
            else:
                current_train_data = train_dataset
            
            # Create dataloaders
            train_loader = DataLoader(
                current_train_data,
                batch_size=self.config.batch_size,
                shuffle=True,
                num_workers=4,
                pin_memory=True
            )
            
            val_loader = DataLoader(
                val_dataset,
                batch_size=self.config.batch_size,
                shuffle=False,
                num_workers=4,
                pin_memory=True
            )
            
            # Train
            train_metrics = self._train_epoch(train_loader, epoch)
            self.train_metrics_history.append(train_metrics)
            
            # Validate
            if epoch % self.config.validate_every == 0:
                val_metrics = self._validate(val_loader)
                self.val_metrics_history.append(val_metrics)

                # Learning rate scheduling
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_metrics[self.config.monitor_metric])
                elif self.scheduler is not None and not isinstance(self.scheduler, optim.lr_scheduler.OneCycleLR):
                    self.scheduler.step()
                
                # Check for improvement and checkpoint
                current_metric = val_metrics[self.config.monitor_metric]
                is_best = self.checkpoint_manager.check_improvement(current_metric)
                
                if is_best:
                    best_metric_value = current_metric
                
                self.checkpoint_manager.save_checkpoint(
                    self.model, self.optimizer, self.scheduler,
                    epoch, val_metrics, is_best=is_best
                )
                
                # Early stopping check
                if self.early_stopping(current_metric):
                    logger.info(f"Early stopping at epoch {epoch}")
                    break
                
                # Logging
                logger.info(
                    f"Epoch {epoch}/{self.config.epochs} | "
                    f"Time: {time.time() - epoch_start_time:.1f}s | "
                    f"Train Loss: {train_metrics['loss']:.4f} | "
                    f"Val {self.config.monitor_metric}: {current_metric:.4f} | "
                    f"Val Seq Acc: {val_metrics['sequence_accuracy']:.4f} | "
                    f"Val BLEU: {val_metrics['bleu']:.4f}"
                )
            else:
                logger.info(f"Epoch {epoch} (no validation) - Train Loss: {train_metrics['loss']:.4f}")
        
        logger.info("Training completed!")
        return self.train_metrics_history, self.val_metrics_history
    
    def save_history(self, path: str):
        """Save training history to JSON"""
        history = {
            'train': self.train_metrics_history,
            'val': self.val_metrics_history,
            'config': self.config.to_dict()
        }
        with open(path, 'w') as f:
            json.dump(history, f, indent=2)
        logger.info(f"Saved training history to {path}")


# Convenience function for quick training setup
def train_model(model, train_dataset, val_dataset=None, 
                config: Optional[TrainingConfig] = None,
                checkpoint_dir: str = "/kaggle/working/checkpoints/"):
    """
    Quick training function with sensible defaults.
    
    Args:
        model: LSTM or Transformer model from Phase 3
        train_dataset: Training data
        val_dataset: Validation data (optional)
        config: Training configuration (uses defaults if None)
        checkpoint_dir: Where to save checkpoints
    
    Returns:
        Trainer instance with trained model
    """
    if config is None:
        config = TrainingConfig(
            checkpoint_dir=checkpoint_dir,
            use_curriculum=True,
            lr_scheduler="onecycle",
            early_stopping_patience=10
        )
    
    # Adjust gradient clip for LSTM specifically
    if hasattr(model, 'model_type') and model.model_type == 'lstm':
        config.gradient_clip_norm = 1.0  # Essential for LSTMs
        logger.info("LSTM detected: Using gradient clipping norm=1.0")
    
    trainer = Trainer(model, config)
    trainer.fit(train_dataset, val_dataset)
    
    return trainer


if __name__ == "__main__":
    # Example usage
    from FASEROH.models.lstm_seq2seq import LSTMTaylorModel
    from FASEROH.models.transformer import TransformerTaylorModel
    from utils import ModelConfig
    
    # Create dummy model for testing
    model_config = ModelConfig(vocab_size=100, d_model=64, max_seq_len=50)
    model = LSTMTaylorModel(model_config)
    
    # Create dummy dataset
    class DummyDataset(torch.utils.data.Dataset):
        def __init__(self, size=1000, seq_len=20):
            self.size = size
            self.seq_len = seq_len
            
        def __len__(self):
            return self.size
            
        def __getitem__(self, idx):
            return {
                'encoder_input': torch.randint(4, 100, (self.seq_len,)),
                'decoder_target': torch.randint(4, 100, (self.seq_len,)),
                'encoder_mask': torch.ones(self.seq_len, dtype=torch.bool)
            }
    
    train_data = DummyDataset(1000)
    val_data = DummyDataset(200)
    
    # Test training
    config = TrainingConfig(
        epochs=2,
        batch_size=512,
        checkpoint_dir="/kaggle/FASEROH/working/test_checkpoints/",
        use_curriculum=False,
        early_stopping_patience=5
    )
    
    trainer = Trainer(model, config)
    trainer.fit(train_data, val_data)

Writing FASEROH/training/utils.py


In [ ]:
%%writefile FASEROH/training/train_lstm.py

import logging
import argparse
from pathlib import Path
import sys
import os
sys.path.append(os.getcwd())

from FASEROH.data.dataset import build_complete_pipeline 
from FASEROH.training.utils import ModelConfig, TrainingConfig, TaylorModelFactory, train_model

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def parse_args():
    """Parse command line arguments for training."""
    parser = argparse.ArgumentParser(description="Train LSTM Model for Taylor Expansions")
    
    # Data arguments
    parser.add_argument("--data_dir", type=str, default="data/raw_datasets", help="Directory containing JSONL data")
    parser.add_argument("--output_dir", type=str, default="data/tokenized_data", help="Where to save tokenized arrays")
    parser.add_argument("--checkpoint_dir", type=str, default="results/lstm_checkpoints", help="Where to save model weights")
    
    # Training arguments
    parser.add_argument("--batch_size", type=int, default=512, help="Training batch size")
    parser.add_argument("--epochs", type=int, default=100, help="Maximum number of training epochs")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
    
    # Model arguments
    parser.add_argument("--d_model", type=int, default=256, help="Embedding dimension")
    parser.add_argument("--lstm_hidden", type=int, default=256, help="LSTM hidden state dimension")
    
    return parser.parse_args()

def main():
    # 1. Grab the arguments from the terminal
    args = parse_args()
    logger.info(f"Setting up LSTM Training Pipeline with args: {args}")

    # 2. Use the args for paths
    data_path = Path(args.data_dir)
    pipeline, metadata = build_complete_pipeline(
        train_jsonl="/kaggle/input/datasets/iamogh/faseroh/train_data.jsonl",
        test_jsonl="/kaggle/input/datasets/iamogh/faseroh/test_data.jsonl",
        output_dir=args.output_dir
    )
    
    train_loader = pipeline.create_pytorch_dataset(split='train')
    val_loader = pipeline.create_pytorch_dataset(split='test')

    # 3. Use the args for Model Config
    model_config = ModelConfig(
        vocab_size=metadata['vocab_size_output'], 
        d_model=args.d_model, 
        lstm_hidden=args.lstm_hidden,
        pad_idx=metadata['special_tokens']['pad']
    )
    model = TaylorModelFactory.create_model("lstm", model_config)

    # 4. Use the args for Training Config
    train_config = TrainingConfig(
        epochs=args.epochs,
        batch_size=args.batch_size,
        learning_rate=args.lr,
        checkpoint_dir=args.checkpoint_dir,
        use_curriculum=True,
        monitor_metric="sequence_accuracy"
    )

    # 5. Execute
    trainer = train_model(model, train_loader, val_loader, train_config)
    
    # Save metrics cleanly
    metrics_path = Path(args.checkpoint_dir).parent / "lstm_metrics.json"
    trainer.save_history(str(metrics_path))

    from FASEROH.evaluation.evaluate import run_full_evaluation
    logger.info("Starting Final Phase 5 Evaluation...")
    run_full_evaluation(model, pipeline.output_tokenizer, val_loader, output_dir="results/lstm_evaluation")

if __name__ == "__main__":
    main()

Writing FASEROH/training/train_lstm.py


In [ ]:
%%writefile FASEROH/training/train_transformer.py

import logging
import argparse
from pathlib import Path
import torch
import sys

import sys
import os
# Dynamically find the project root instead of hardcoding /teamspace/
sys.path.append(os.getcwd())

# 1. Import your Phase 2 Data Pipeline
from FASEROH.data.dataset import build_complete_pipeline 

# 2. Import your Phase 3 & 4 Configs, Factory, and Trainer
from FASEROH.training.utils import ModelConfig, TrainingConfig, TaylorModelFactory, train_model

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def parse_args():
    """Parse command line arguments for Transformer training."""
    parser = argparse.ArgumentParser(description="Train Transformer Model for Taylor Expansions")
    
    # Data and Path arguments
    parser.add_argument("--data_dir", type=str, default="data/raw_datasets", help="Directory containing JSONL data")
    parser.add_argument("--output_dir", type=str, default="data/tokenized_data", help="Where to save tokenized arrays")
    parser.add_argument("--checkpoint_dir", type=str, default="results/transformer_checkpoints", help="Where to save model weights")
    
    # Training arguments
    parser.add_argument("--batch_size", type=int, default=32, help="Training batch size")
    parser.add_argument("--epochs", type=int, default=100, help="Maximum number of training epochs")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
    parser.add_argument("--label_smoothing", type=float, default=0.1, help="Label smoothing factor")
    
    # Transformer-specific arguments
    parser.add_argument("--d_model", type=int, default=256, help="Embedding dimension")
    parser.add_argument("--n_layers", type=int, default=4, help="Number of encoder/decoder layers")
    parser.add_argument("--n_heads", type=int, default=8, help="Number of attention heads")
    parser.add_argument("--d_ff", type=int, default=1024, help="Feed-forward network dimension")
    parser.add_argument("--pos_encoding", type=str, default="sinusoidal", choices=["sinusoidal", "learned"], help="Type of positional encoding")
    
    return parser.parse_args()

def main():
    # 1. Grab the arguments from the terminal
    args = parse_args()
    logger.info(f"Setting up Transformer Training Pipeline with args: {args}")

    # 2. Setup Data Pipeline
    data_path = Path(args.data_dir)
    pipeline, metadata = build_complete_pipeline(
        train_jsonl="/kaggle/input/datasets/iamogh/faseroh/train_data.jsonl",
        test_jsonl="/kaggle/input/datasets/iamogh/faseroh/test_data.jsonl",
        output_dir=args.output_dir
    )
    
    train_loader = pipeline.create_pytorch_dataset(split='train')
    val_loader = pipeline.create_pytorch_dataset(split='test')

    # 3. Configure and Build the Transformer
    model_config = ModelConfig(
        vocab_size=metadata['vocab_size_output'], 
        d_model=args.d_model, 
        n_layers=args.n_layers,
        n_heads=args.n_heads,
        d_ff=args.d_ff,
        positional_encoding=args.pos_encoding,
        pad_idx=metadata['special_tokens']['pad']
    )
    
    # The factory fetches the Transformer architecture
    model = TaylorModelFactory.create_model("transformer", model_config)

    # 4. Configure Training Strategy
    train_config = TrainingConfig(
        epochs=args.epochs,
        batch_size=args.batch_size,
        learning_rate=args.lr,
        label_smoothing=args.label_smoothing,
        checkpoint_dir=args.checkpoint_dir,
        use_curriculum=True,
        monitor_metric="loss"  # The safe metric key we fixed earlier
    )

    trainer = train_model(model, train_loader, val_loader, train_config)
    
    # Save the final metrics to a JSON file
    metrics_path = Path(args.checkpoint_dir).parent / "transformer_metrics.json"
    trainer.save_history(str(metrics_path))

    from FASEROH.evaluation.evaluate import run_full_evaluation
    logger.info("Starting Final Phase 5 Evaluation...")
    run_full_evaluation(model, pipeline.output_tokenizer, val_loader, output_dir="results/lstm_evaluation")

if __name__ == "__main__":
    main()

Writing FASEROH/training/train_transformer.py


In [ ]:
!python3 FASEROH/training/train_lstm.py --output_dir "/kaggle/working/processed_data" --checkpoint_dir "/kaggle/working/checkpoints" --batch_size 128 --epochs 100 --lr 0.002

2026-03-27 15:14:29,693 - numexpr.utils - INFO - NumExpr defaulting to 4 threads.
2026-03-27 15:14:32,193 - __main__ - INFO - Setting up LSTM Training Pipeline with args: Namespace(data_dir='data/raw_datasets', output_dir='/kaggle/working/processed_data', checkpoint_dir='/kaggle/working/checkpoints', batch_size=128, epochs=100, lr=0.002, d_model=256, lstm_hidden=256)
2026-03-27 15:14:32,606 - FASEROH.data.dataset - INFO - Fitting input tokenizer...
2026-03-27 15:14:32,607 - FASEROH.data.tokenizer - INFO - Building vocabulary (streaming mode)...
2026-03-27 15:14:32,684 - FASEROH.data.tokenizer - INFO -   Scanned 10000 expressions...
2026-03-27 15:14:32,759 - FASEROH.data.tokenizer - INFO -   Scanned 20000 expressions...
2026-03-27 15:14:32,831 - FASEROH.data.tokenizer - INFO -   Scanned 30000 expressions...
2026-03-27 15:14:32,849 - FASEROH.data.tokenizer - INFO - Vocabulary built: 30 tokens
2026-03-27 15:14:32,849 - FASEROH.data.tokenizer - INFO - Top 10 tokens: [('*', 86293), ('x', 67

In [ ]:
import shutil
import os

# 1. Define what you want to zip (the results folder)
# This matches the output_dir and checkpoint_dir from your script
output_path = "/kaggle/working/faseroh_final_results"
source_dir = "/kaggle/working/" 

# 2. Create the zip archive
shutil.make_archive(output_path, 'zip', source_dir)

print(f"✅ Archive created: {output_path}.zip")